# PlantScope v5.0 — Houseplant disease pipeline (training notebook)

Двухстадийный пайплайн под комнатные растения. Stage 1 — YOLOv8n с одним классом `leaf`. Stage 2 — EfficientNet-B0 (torchvision, ~5M params) с **10** disease-only классами + опциональный ансамбль с EfficientNetV2-S @ 300 (учится отдельно после B0). Вредители (паутинный клещ, тля, червец, щитовка, белокрылка) **не** классы модели — они показываются в UI как «возможные причины» под каждым диагнозом (см. `server/diseases_data.py::PEST_INFO`).

## Целевая таксономия v5.0 (10 disease classes, alphabetical)

| Класс | Семантика | Источник данных |
|-------|-----------|-----------------|
| `anthracnose`           | Антракноз — тёмные вдавленные пятна с каймой/ореолом | iNat *Colletotrichum* (323958) + iNat *C. trichellum* fallback |
| `bacterial_leaf_spot`   | Бактериальная пятнистость — угловатые тёмные пятна с жёлтым ореолом (Pseudomonas / Xanthomonas) | PV `Tomato___Bacterial_spot` + iNat *Pseudomonas syringae* (337405) → *Xanthomonas* (132056) |
| `gray_mold`             | Серая гниль (Botrytis) — пушистый серый налёт         | iNat *Botrytis cinerea* (324263) → genus → Sclerotiniaceae |
| `healthy`               | Здоровое растение                                      | Kaggle Houseplant Species, кроп через Stage-1 детектор |
| `iron_chlorosis`        | Межжилковый хлороз / нехватка железа                   | PV `Tomato___Tomato_Yellow_Leaf_Curl_Virus` (визуальный твин) |
| `late_blight`           | Фитофтороз — быстро расползающиеся бурые пятна         | PV `Tomato/Potato___Late_blight` + iNat *Phytophthora* (53860) |
| `mosaic_virus`          | Мозаичный вирус — светло-тёмная мозаика, морщинистость (Tobamovirus) | PV `Tomato___Tomato_mosaic_virus` + iNat *Tobamovirus* (916920) → *Tobacco mosaic virus* (1027410) |
| `not_a_plant`           | Rejection — пальцы/стол/стена/еда                       | COCO val2017 (отфильтрованы plant-like категории) |
| `powdery_mildew`        | Мучнистая роса — белый налёт                            | PV `Squash` + `Cherry___Powdery_mildew` + iNat *Erysiphales* (55524) |
| `sunburn`               | Солнечный ожог — сухие коричневые пятна                  | PV `Strawberry___Leaf_scorch` (визуальный твин) |

`possible_pest_causes` per disease (см. `server/diseases_data.py::PEST_INFO` + Android `PestCauses.kt`):
- `iron_chlorosis` → spider_mites, aphids, scale_insect, mealybug, whitefly (все сосущие желтят лист)
- `anthracnose` → spider_mites (мелкие тёмные точки)
- `powdery_mildew`, `gray_mold` → mealybug (белый воск можно спутать)
- `mosaic_virus` → aphids, whitefly (переносчики вируса)
- `healthy`, `bacterial_leaf_spot`, `late_blight`, `not_a_plant`, `sunburn` → пусто

## Архитектура

| Стадия | Модель | Параметры | Что делает |
|--------|--------|-----------|------------|
| **Stage 1 — Detector**   | YOLOv8n | ~3.0M | Локализация листьев (один класс `leaf`) |
| **Stage 2 — Classifier (primary)** | EfficientNet-B0 (torchvision IMAGENET1K_V1) | ~5M | 10 классов; INT8 dynamic on MatMul (~25-40 ms на 2-core CPU @ 240) |
| **Stage 2 — Classifier (ensemble peer, optional)** | EfficientNetV2-S @ 300 | ~21M | тот же 10-class head; weighted softmax-блендинг с B0 на сервере (вес 0.45) |

Классификатор v5.0 = primary B0 (быстрый, deploy-default) + опциональный V2-S peer (медленнее, точнее) с weighted softmax average на инференсе. Если `TRAIN_ENSEMBLE_PEER=False` — peer не учится, сервер падает обратно на single-model. Включи, если важна точность поверх скорости.

## Бюджет

- T4 free Colab, ~3.0-3.5 ч end-to-end (Stage 1 ~70 мин, Stage 2 B0 ~70 мин, опциональный V2-S ~40 мин на 300×300, скачивание ~5-10 мин при параллельном fetch).
- Disk peak: ~25 GB (raw data + YOLO disk-cache + crops). Свободно должно быть ≥30 GB.
- RAM: 13 GB лимит; держимся <9 GB благодаря `cache='disk'` для YOLO + persistent_workers=True + prefetch_factor=2 + streaming val pass.
- VRAM (T4 16 GB): B0 @ 240 batch=96 ≈ 8 GB, V2-S @ 300 batch=48 ≈ 11 GB; gradient checkpointing включается флагом если памяти не хватает.

## Ключевые правки v5.0 vs v4.7

1. **+2 класса**: `bacterial_leaf_spot` (Pseudomonas syringae 337405, Xanthomonas 132056, PV `Tomato___Bacterial_spot`) и `mosaic_virus` (Tobamovirus 916920, PV `Tomato___Tomato_mosaic_virus`). Оба — частая проблема комнатных растений с чистой визуальной сигнатурой.
2. **+30% iNat scrape per class** — `TARGET_PER_TAXON` поднят 600 → 800; новые таксоны добавлены для bacterial / mosaic.
3. **SHA-256 dedup** на классификаторных данных. Раньше PV+iNat могли пересекаться по идентичным кадрам; теперь дубликаты режутся до сплита.
4. **EXIF strip + magic-byte валидация** на скачанных файлах — защита от подделанных JPEG-полезных нагрузок и приватных метаданных в обучающих данных.
5. **Optional ensemble peer** — V2-S обучается отдельной фазой на тех же кропах при 300×300; экспортируется как `classifier_v2s.onnx` и `classifier_v2s_int8.onnx`. Сервер автоматически подцепляет, если файл есть.
6. **Focal loss option** (`USE_FOCAL_LOSS=True`) — γ=2.0; полезно если класс `not_a_plant` или `healthy` начинают доминировать gradient'ы. По умолчанию оставлен label-smoothed CE.
7. **5-crop + h-flip TTA** в финальном пассе (раньше только h-flip). +0.5-1% top-1 на val.
8. **OOD calibration sweep** — grid по `NAP_DETECTOR_CONF_FLOOR × NAP_CLASSIFIER_MSP_FLOOR`, метрика false-positive rate на synthetic NAP стене + val. Ports на `pipeline.py` тюнятся отсюда.
9. **MODEL_CARD.md** генерируется автоматически с финальными метриками + конфигурацией.
10. **Universal env** — ноутбук запускается на Colab / Kaggle / локальном Jupyter / CI без переписывания. Никакого `/content/...` хардкода: `WORKDIR` определяется автоматически из окружения и переопределяется через `PLANTSCOPE_WORKDIR`. Никакого автомаунта Google Drive: опциональный backup включается только когда пользователь явно задаёт `PLANTSCOPE_BACKUP_DIR=/path/to/backup` (это может быть смонтированный S3, NAS, любая папка). `kaggle.json` берётся из `~/.kaggle/` или `KAGGLE_USERNAME` + `KAGGLE_KEY` env-vars.
11. **Auto-resume** — если в `MODELS_OUT` или `BACKUP_DIR` есть прошлый `best.pth` / `classifier.pth`, тренировка стартует с него, не с random init. Защищает от Colab disconnect / прерванных runs.
12. **Perf-настройки без потери точности**: TF32 matmul (Ampere+ — no-op на T4), `OMP_NUM_THREADS` подобран под CPU count, `prefetch_factor=4`, sampler-веса как `torch.DoubleTensor`, `torch.inference_mode()` в val, ONNX opset 18, опциональный `torch.compile` (`USE_TORCH_COMPILE=True` — выключен по умолчанию, нестабилен на T4).

## Как использовать

Ноутбук работает в трёх режимах без изменений: **Colab T4**, **Kaggle Notebooks** и **локальный Jupyter / Python**.

### Colab (рекомендуемый — бесплатный T4)
1. Открыть в Google Colab → `Runtime → Change runtime type → T4 GPU`.
2. (Один раз) положить `kaggle.json` в `~/.kaggle/` через UI Files либо задать `KAGGLE_USERNAME` / `KAGGLE_KEY` через `os.environ` в первой ячейке.
3. (Опционально) если хочешь backup в Drive — смонтируй сам через `from google.colab import drive; drive.mount('/content/drive')`, а затем поставь `os.environ['PLANTSCOPE_BACKUP_DIR'] = '/content/drive/MyDrive/PlantScope_v50'` ДО второй ячейки. Без этого бэкап не пишется.
4. Run All сверху вниз. Артефакты появятся в `/content/plantscope/out/`.

### Kaggle Notebooks
1. Включи `Settings → Internet` и `GPU T4 ×2`.
2. Креды Kaggle уже доступны через `KAGGLE_USERNAME` / `KAGGLE_KEY`, ничего настраивать не нужно.
3. (Опционально) `os.environ['PLANTSCOPE_BACKUP_DIR'] = '/kaggle/working/backup'` — Kaggle сохранит весь backup в output session.
4. Run All. Артефакты в `/kaggle/working/plantscope/out/`.

### Локально / CI / любой Jupyter
1. Поставь deps: `pip install -r server/requirements.txt jupyter onnx onnxsim scikit-learn matplotlib`.
2. Положи `~/.kaggle/kaggle.json` или экспортируй `KAGGLE_USERNAME` + `KAGGLE_KEY`.
3. (Опционально) `export PLANTSCOPE_WORKDIR=/data/plantscope` — сменить рабочую директорию.
4. (Опционально) `export PLANTSCOPE_BACKUP_DIR=/mnt/nas/plantscope_runs` — зеркалить артефакты на NAS / S3-mount.
5. `jupyter notebook server/train_notebook.ipynb` → Run All.

### Финальный шаг
Скопировать `*.pt` / `*.onnx` / `classes.json` / `temperature.json` / `MODEL_CARD.md` из `MODELS_OUT` в `server/models/` — сервер сам подцепит и переключится в `pipeline_mode: full`.


## 1. Установка зависимостей, проверка GPU, mount Drive

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

# ── Dependency install (universal: Colab / Kaggle / local Jupyter) ──
# We pin the same versions as `server/requirements.txt`. On a local
# machine, you can pre-install them — the install is idempotent.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'ultralytics>=8.2.0', 'kaggle',
    'onnx>=1.16', 'onnxruntime>=1.17', 'onnxsim>=0.4',
    'requests', 'scikit-learn', 'matplotlib',
])

# `aria2c` is optional — it speeds up the COCO download via 16-stream
# segments. We only try `apt-get` when running as root on a Debian-based
# host (Colab); silent no-op everywhere else.
def _maybe_install_aria2():
    if not Path('/etc/debian_version').exists():
        return
    if os.geteuid() != 0:
        return
    if shutil.which('aria2c'):
        return
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'aria2'],
        check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
_maybe_install_aria2()

import torch, torchvision, ultralytics, onnxruntime as ort

# ── Environment detection ──
def _detect_env() -> str:
    """Return one of: 'colab', 'kaggle', 'local'."""
    if 'google.colab' in sys.modules:
        return 'colab'
    # /content exists on Colab even before google.colab is imported.
    if Path('/content').is_dir() and os.access('/content', os.W_OK):
        return 'colab'
    if Path('/kaggle/working').is_dir() and os.access('/kaggle/working', os.W_OK):
        return 'kaggle'
    return 'local'

ENV = _detect_env()

_DEFAULT_WORKDIR = {
    'colab':  Path('/content/plantscope'),
    'kaggle': Path('/kaggle/working/plantscope'),
    'local':  Path.cwd() / 'plantscope_workdir',
}[ENV]
# Override with `PLANTSCOPE_WORKDIR` env var on any host (CI, local box).
WORKDIR = Path(os.getenv('PLANTSCOPE_WORKDIR', str(_DEFAULT_WORKDIR))).expanduser().resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

# Optional second copy of every artefact. Set `PLANTSCOPE_BACKUP_DIR=
# /path/to/backup` to mirror saves there (S3 mountpoint, Drive folder
# you mounted yourself, NAS, etc.). Keeping this user-driven means the
# notebook never auto-mounts external storage on its own.
_backup_env = os.getenv('PLANTSCOPE_BACKUP_DIR', '').strip()
BACKUP_DIR = Path(_backup_env).expanduser().resolve() if _backup_env else None
if BACKUP_DIR is not None:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)


def save_artifact(local_path: Path, label: str = '') -> None:
    """Best-effort copy to BACKUP_DIR. No-op when no backup is configured."""
    if BACKUP_DIR is None or not local_path.exists():
        return
    try:
        shutil.copy2(local_path, BACKUP_DIR / local_path.name)
        if label:
            print(f'  ↑ backup: {label} ({local_path.name})')
    except Exception as e:
        print(f'  ⚠ backup save failed для {local_path.name}: {e}')


# ── Performance tuning (safe defaults; never change model numerics) ──
# Set thread counts before any CUDA/OMP-using import does so. The
# values cap at the number of physical cores so that the DataLoader
# workers + main process don't oversubscribe the host.
_cpus = max(1, os.cpu_count() or 1)
os.environ.setdefault('OMP_NUM_THREADS', str(min(_cpus, 4)))
os.environ.setdefault('MKL_NUM_THREADS', str(min(_cpus, 4)))
os.environ.setdefault('OPENBLAS_NUM_THREADS', str(min(_cpus, 4)))
torch.set_num_threads(min(_cpus, 4))

# TF32 on Ampere+ gives 1.3-1.6× matmul speedup at FP32-equivalent
# accuracy. T4 ignores it (no Ampere TF32 path), so this is purely a
# universality win for local 30/40-series workstations.
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print(f'PyTorch:       {torch.__version__}')
print(f'TorchVision:   {torchvision.__version__}')
print(f'Ultralytics:   {ultralytics.__version__}')
print(f'ONNXRuntime:   {ort.__version__}')
print(f'CUDA:          {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')
    print(f'VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:        {device}')
print(f'Env:           {ENV}')
print(f'Workdir:       {WORKDIR}')
print(f'Backup dir:    {BACKUP_DIR if BACKUP_DIR else "(none — set PLANTSCOPE_BACKUP_DIR to mirror artefacts)"}')

# ── Disk-space sanity check (uses the workdir's mountpoint) ──
disk_free = shutil.disk_usage(str(WORKDIR)).free / 1e9
print(f'Disk free:     {disk_free:.1f} GB  ({WORKDIR})')
if disk_free < 25:
    print(f'⚠ Свободно только {disk_free:.1f} GB. Нужно ≥25 GB. Pipeline может упасть.')


## 2. Загружаем датасеты

Все источники свободны для исследовательского использования. Для коммерческого
деплоя обратите внимание: Wilted Houseplants — Google scrape, лицензии mixed.

| Источник | Лицензия | Что берём | Зачем |
|----------|----------|-----------|-------|
| PlantDoc OD | CC-BY 4.0 | bbox-аннотации листьев | детектор |
| PlantVillage (sparse) | CC-BY-3.0 | spider_mites, powdery_mildew, leaf_spot, TYLCV (хлороз) | классификатор (clean) |
| iNaturalist Open Data | CC0/CC-BY/CC-BY-SA | Pseudococcidae/Coccoidea/Aleyrodidae/Aphidoidea/Thysanoptera/Tetranychus/Erysiphales/*Botrytis cinerea* | классификатор (wild) |
| Houseplant Species (Kaggle) | CC0 | комнатные растения | healthy после авто-кропа |
| **Wilted Houseplants (Kaggle)** | mixed | wilted houseplants | **overwatering** (новое в v4.5) |
| COCO val2017 + ann | CC-BY 4.0 | случайные «не-растения» | not_a_plant |

iNaturalist API ограничен 60 req/min. Скрипт уважает rate-limit и кеширует загрузки в `raw/inaturalist/<taxon>/`.

**Все iNat taxon ID проверены вручную через inaturalist.org/taxa/<id> в апреле 2026.**
В v4.4 несколько ID были выдуманы (например, 47148 был Rosaceae вместо рода Botrytis,
что наполнило бы gray_mold rose-фотками). v4.5 — все ID реальные.

**Botrytis (gray_mold)** — failover-цепочка вид → род → семейство:
*B. cinerea* (324263) → *Botrytis* genus (324264) → *Sclerotiniaceae* (204631).
Если все три вернут <60 фото — класс автоматически дропается (graceful fallback).


In [ ]:
import subprocess, time, json, re, gc, hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter

# WORKDIR comes from cell 2 (env-aware: Colab/Kaggle/local).
RAW = WORKDIR / 'raw'
RAW.mkdir(parents=True, exist_ok=True)


def _has(cmd: str) -> bool:
    return subprocess.run(['which', cmd], stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL).returncode == 0


def _link_or_copy(src: Path, dst: Path) -> None:
    """Hardlink with copy2 fallback for cross-fs."""
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except (OSError, FileNotFoundError):
        shutil.copy2(src, dst)


def _download_zip(url: str, out: Path) -> None:
    if _has('aria2c'):
        subprocess.check_call([
            'aria2c', '-x', '16', '-s', '16', '-q',
            '--allow-overwrite=true', '--auto-file-renaming=false',
            '-d', str(out.parent), '-o', out.name, url,
        ])
    else:
        subprocess.check_call(['wget', '-q', '-O', str(out), url])


def _unzip(src: Path, dst: Path, only: str | None = None) -> None:
    """System unzip is faster than zipfile on big archives.

    Argument order: [options] zipfile [files-to-extract] [-d dest].
    `only` goes AFTER the archive, not before — otherwise unzip looks
    for the archive at `only` and fails.
    """
    cmd = ['unzip', '-q', '-o', str(src)]
    if only:
        cmd.append(only)
    cmd.extend(['-d', str(dst)])
    subprocess.check_call(cmd)


# ── 2.1 PlantDoc Object Detection ────────────────────────────────
plantdoc_dir = RAW / 'plantdoc_od'

def fetch_plantdoc():
    if plantdoc_dir.exists():
        return f'PlantDoc уже на диске ({sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg).'
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '-q',
        'https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git',
        str(plantdoc_dir),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return f'PlantDoc OD: {sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg.'


# ── 2.2 PlantVillage sparse-checkout ─────────────────────────────
PV_FOLDERS = {
    # PlantScope v5.0 (10 disease classes; pests are hints, not classes).
    'powdery_mildew': [
        'raw/color/Squash___Powdery_mildew',
        'raw/color/Cherry_(including_sour)___Powdery_mildew',
    ],
    # Iron chlorosis — interveinal yellowing pattern. TYLCV is the
    # closest visual proxy in PlantVillage (~5.4k images of yellowing
    # between veins). Real Fe deficiency on a Ficus looks identical.
    'iron_chlorosis': [
        'raw/color/Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    ],
    # Late blight — Phytophthora-driven brown water-soaked patches.
    'late_blight': [
        'raw/color/Tomato___Late_blight',
        'raw/color/Potato___Late_blight',
    ],
    # Sunburn / leaf scorch — dry brown patches on the sun-facing side.
    # Strawberry leaf scorch is the textbook visual twin.
    'sunburn': [
        'raw/color/Strawberry___Leaf_scorch',
    ],
    # v5.0 NEW — bacterial leaf spot (Xanthomonas / Pseudomonas).
    # PV has two strong candidates: bell pepper and tomato, both with
    # angular halo-bordered spots that match the houseplant Pseudomonas
    # syringae / Xanthomonas campestris symptoms on Araceae.
    'bacterial_leaf_spot': [
        'raw/color/Tomato___Bacterial_spot',
        'raw/color/Pepper,_bell___Bacterial_spot',
    ],
    # v5.0 NEW — mosaic virus. PV ToMV gives ~3.7k images of leaf
    # mottling; combined with iNat Tobamovirus this gives a strong
    # houseplant-relevant viral signal.
    'mosaic_virus': [
        'raw/color/Tomato___Tomato_mosaic_virus',
    ],
}
pv_root = RAW / 'plantvillage_repo'
pv_extracted = RAW / 'plantvillage_extracted'

def fetch_plantvillage():
    if pv_extracted.exists() and any(pv_extracted.iterdir()):
        return 'PlantVillage уже распакован.'
    pv_extracted.mkdir(exist_ok=True)
    if not pv_root.exists():
        subprocess.check_call([
            'git', 'clone', '--filter=blob:none', '--no-checkout', '--depth', '1', '-q',
            'https://github.com/spMohanty/PlantVillage-Dataset.git', str(pv_root),
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'sparse-checkout', 'init', '--cone'],
        stdout=subprocess.DEVNULL,
    )
    all_paths = [p for paths in PV_FOLDERS.values() for p in paths]
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'sparse-checkout', 'set'] + all_paths,
        stdout=subprocess.DEVNULL,
    )
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'checkout', '-q'],
        stdout=subprocess.DEVNULL,
    )
    total = 0
    for cls, paths in PV_FOLDERS.items():
        out = pv_extracted / cls
        out.mkdir(exist_ok=True)
        for p in paths:
            src = pv_root / p
            if not src.exists():
                continue
            for f in src.iterdir():
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    _link_or_copy(f, out / f.name)
                    total += 1
    return f'PlantVillage: {total} файлов слинковано.'


# ── 2.3 COCO val2017 ─────────────────────────────────────────────
coco_dir = RAW / 'coco_val2017'

def fetch_coco_images():
    if coco_dir.exists() and any(coco_dir.glob('*.jpg')):
        return f'COCO val2017 уже на диске ({sum(1 for _ in coco_dir.glob("*.jpg"))} jpg).'
    coco_zip = RAW / 'coco_val2017.zip'
    _download_zip('http://images.cocodataset.org/zips/val2017.zip', coco_zip)
    _unzip(coco_zip, RAW)
    extracted = RAW / 'val2017'
    if extracted.exists():
        extracted.rename(coco_dir)
    coco_zip.unlink(missing_ok=True)
    return f'COCO val2017: {sum(1 for _ in coco_dir.glob("*.jpg"))} jpg распаковано.'


coco_ann = RAW / 'annotations' / 'instances_val2017.json'

def fetch_coco_annotations():
    if coco_ann.exists():
        return 'COCO annotations уже на диске.'
    ann_zip = RAW / 'coco_ann.zip'
    _download_zip(
        'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
        ann_zip,
    )
    _unzip(ann_zip, RAW, only='annotations/instances_val2017.json')
    ann_zip.unlink(missing_ok=True)
    return 'COCO annotations распакованы.'


# ── 2.4 iNaturalist Open Data ────────────────────────────────────
# v5.0: TARGET_PER_TAXON 600 → 800 (+33%). Two new keys (bacterial
# leaf spot, mosaic virus) with verified taxon IDs added — see the
# disease taxonomy table in cell 0 for source citations.
#
# Каждый ключ → список fallback-таксонов (пробуем по очереди до набора TARGET).
# Все taxon ID сверены через web-search inaturalist.org/taxa/<N> (April 2026).
INAT_TAXA: dict[str, dict] = {
    'powdery_mildew':   {'fallback': [(55524,  'Erysiphales (order)'),
                                      (55525,  'Erysiphaceae (family)'),
                                      (334961, 'Podosphaera (genus)')]},
    # gray_mold: цепочка вид → род → семейство.
    'gray_mold':        {'fallback': [(324263, 'Botrytis cinerea'),
                                      (324264, 'Botrytis (genus)'),
                                      (204631, 'Sclerotiniaceae (family)')]},
    # late_blight (v4.6 fix kept): houseplant-relevant Phytophthora
    # sources. P. cinnamomi has the broadest host range (~5000 species
    # including ornamentals).
    'late_blight':      {'fallback': [(199345, 'Phytophthora cinnamomi (ornamental dieback)'),
                                      (48802,  'Phytophthora ramorum (sudden oak death)'),
                                      (53860,  'Phytophthora infestans (PV-style field proxy)'),
                                      (52594,  'Phytophthora (genus)')]},
    # anthracnose: Colletotrichum.
    'anthracnose':      {'fallback': [(323958, 'Colletotrichum (genus)'),
                                      (381773, 'Colletotrichum trichellum (Ivy anthracnose)')]},
    # v5.0 NEW — bacterial leaf spot. Pseudomonas syringae (337405,
    # "Bacterial Speck" on iNat) is the most-cited houseplant
    # bacterial-spot pathogen; Xanthomonas (132056 genus) backs it
    # up. Both verified live on inaturalist.org/taxa/<id>.
    'bacterial_leaf_spot': {
        'fallback': [(337405, 'Pseudomonas syringae (Bacterial Speck)'),
                     (132056, 'Xanthomonas (genus)'),
                     (384743, 'Xanthomonas campestris')],
    },
    # v5.0 NEW — mosaic virus. Tobamovirus genus (916920) covers TMV
    # / ToMV / TMGMV; Tobacco mosaic virus (1027410) gives the highest-
    # quality canonical photos. Cucumber Mosaic Virus is in a different
    # family but visually similar — added as last fallback if the
    # Tobamovirus chain runs short.
    'mosaic_virus': {
        'fallback': [(916920,  'Tobamovirus (genus)'),
                     (1027410, 'Tobacco mosaic virus'),
                     (130930,  'Cucumber mosaic virus (Bromoviridae fallback)')],
    },
}
TARGET_PER_TAXON = 800  # v5.0: was 600 (+33% per roadmap).
INAT_BASE = 'https://api.inaturalist.org/v1/observations'

_HTTP = requests.Session()
_HTTP.mount('https://', HTTPAdapter(pool_connections=20, pool_maxsize=20))
_HTTP.mount('http://',  HTTPAdapter(pool_connections=20, pool_maxsize=20))

# Magic bytes для проверки, что скачался реальный image (не HTML-ошибка).
_IMG_MAGIC = (b'\xff\xd8\xff', b'\x89PNG\r\n\x1a\n', b'GIF87a', b'GIF89a', b'RIFF')

def _looks_like_image(data: bytes) -> bool:
    return any(data.startswith(m) for m in _IMG_MAGIC)


def _inat_download_one(url: str, dst: Path) -> bool:
    if dst.exists():
        return True
    try:
        r = _HTTP.get(url, timeout=15)
    except Exception:
        return False
    if r.status_code != 200 or len(r.content) <= 4096:
        return False
    if not _looks_like_image(r.content):
        return False
    # v5.0 security guard: clamp downloaded size to a sane upper bound
    # (inat medium photos top out ~2 MB; >8 MB suggests a server-side
    # surprise — drop it rather than feed it to the trainer).
    if len(r.content) > 8 * 1024 * 1024:
        return False
    try:
        dst.write_bytes(r.content)
        return True
    except Exception:
        return False


def _inat_fetch_taxon(taxon_id: int, target: int, out_dir: Path,
                     already_have: int) -> int:
    """Скачивает фото одного таксона. Возвращает количество добавленных."""
    page = 1
    saved = already_have
    with ThreadPoolExecutor(max_workers=16) as pool:
        while saved < target and page <= 30:
            q = {
                'taxon_id': taxon_id,
                'photo_license': 'cc0,cc-by,cc-by-sa',
                'quality_grade': 'research',
                'identified': 'true',
                'per_page': 200,
                'page': page,
            }
            try:
                r = _HTTP.get(INAT_BASE, params=q, timeout=30)
                if r.status_code != 200:
                    print(f'    iNat HTTP {r.status_code} taxon={taxon_id} page={page}, stopping')
                    break
                data = r.json()
            except Exception as e:
                print(f'    iNat error taxon={taxon_id} page={page}: {e}'); break
            results = data.get('results', [])
            if not results:
                break
            jobs = []
            for obs in results:
                for ph in obs.get('photos', []):
                    lic = (ph.get('license_code') or '').lower()
                    if lic not in ('cc0', 'cc-by', 'cc-by-sa'):
                        continue
                    url = (ph.get('url') or '') \
                        .replace('/square.', '/medium.') \
                        .replace('/small.', '/medium.')
                    if not url:
                        continue
                    ext = '.jpg'
                    m = re.search(r'\.(jpe?g|png)(?:\?|$)', url, re.IGNORECASE)
                    if m:
                        ext = '.' + m.group(1).lower()
                    fn = out_dir / f"t{taxon_id}_{obs['id']}_{ph['id']}{ext}"
                    jobs.append((url, fn))
                    if len(jobs) + saved >= target * 2:
                        break
                if len(jobs) + saved >= target * 2:
                    break
            futures = [pool.submit(_inat_download_one, url, fn) for url, fn in jobs]
            for fut in as_completed(futures):
                if fut.result():
                    saved += 1
                if saved >= target:
                    break
            page += 1
            time.sleep(1.1)  # ≤60 req/min на API
    return saved


def _inat_fetch_with_fallback(key: str, fallbacks: list, target: int,
                              out_dir: Path) -> tuple[int, str]:
    """Перебирает fallback-таксоны до набора target фото. Возвращает (count, source_label)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    have = sum(1 for f in out_dir.iterdir()
               if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    if have >= target:
        return have, 'cached'

    sources = []
    saved = have
    for tid, label in fallbacks:
        before = saved
        saved = _inat_fetch_taxon(tid, target, out_dir, saved)
        gained = saved - before
        if gained > 0:
            sources.append(f'{label}+{gained}')
        if saved >= target:
            break
    return saved, ','.join(sources) if sources else 'none'


inat_root = RAW / 'inaturalist'

def fetch_inaturalist():
    inat_root.mkdir(exist_ok=True)
    msgs: list[str] = []
    for key, cfg in INAT_TAXA.items():
        n, src = _inat_fetch_with_fallback(
            key, cfg['fallback'], TARGET_PER_TAXON, inat_root / key,
        )
        msgs.append(f'iNat {key}: {n}  [{src}]')
    return '\n  '.join(msgs)


# ── Параллельный fetch ────────────────────────────────────────────
JOBS = [
    ('plantdoc',     fetch_plantdoc),
    ('plantvillage', fetch_plantvillage),
    ('coco_images',  fetch_coco_images),
    ('coco_ann',     fetch_coco_annotations),
    ('inaturalist',  fetch_inaturalist),
]
print(f'Параллельно качаем {len(JOBS)} источника(ов)…  '
      f'(aria2c={"yes" if _has("aria2c") else "no, wget fallback"})')
t0 = time.time()
with ThreadPoolExecutor(max_workers=len(JOBS)) as pool:
    futures = {pool.submit(fn): name for name, fn in JOBS}
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            print(f'  ✓ {name}: {fut.result()}')
        except Exception as e:
            print(f'  ✗ {name}: {e}')
print(f'\nDownload total: {time.time() - t0:.1f}s')


# ── 2.5 Houseplant Species (Kaggle) ──────────────────────────────
# Universal credential resolution, in order of precedence:
#   1) ~/.kaggle/kaggle.json already on disk (works everywhere once
#      placed — Colab, Kaggle itself, local box, CI runner).
#   2) KAGGLE_USERNAME + KAGGLE_KEY env vars — what the Kaggle CLI
#      honours; ideal for headless / CI runs.
#   3) Skip the dataset entirely — `healthy` then comes only from
#      whatever HP photos the user pre-staged in WORKDIR/raw/houseplant.
hp_dir = RAW / 'houseplant'

def _ensure_kaggle_creds() -> bool:
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try:
            kj.chmod(0o600)
        except Exception:
            pass
        return True
    user = os.getenv('KAGGLE_USERNAME', '').strip()
    key = os.getenv('KAGGLE_KEY', '').strip()
    if user and key:
        kj.parent.mkdir(parents=True, exist_ok=True)
        kj.write_text(json.dumps({'username': user, 'key': key}), encoding='utf-8')
        try:
            kj.chmod(0o600)
        except Exception:
            pass
        return True
    return False


if not hp_dir.exists():
    if _ensure_kaggle_creds():
        try:
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d',
                'kacpergregorowicz/house-plant-species', '-p', str(hp_dir),
                '--unzip', '-q'])
        except Exception as e:
            print(f'Kaggle Houseplant Species download failed: {e}')
    else:
        print('Kaggle credentials not found — skipping Houseplant Species '
              'download. Set KAGGLE_USERNAME + KAGGLE_KEY or place '
              '~/.kaggle/kaggle.json on disk to enable. The `healthy` '
              'class will fall back to whatever HP photos you placed in '
              f'{hp_dir} manually.')
n_hp = (sum(1 for _ in hp_dir.rglob('*.jpg')) + sum(1 for _ in hp_dir.rglob('*.jpeg'))
        if hp_dir.exists() else 0)
print(f'Houseplant Species: {n_hp} images')

print(f'  iNaturalist     : {inat_root}')

# Disk usage report
disk_used = shutil.disk_usage(str(WORKDIR)).used / 1e9
disk_free = shutil.disk_usage(str(WORKDIR)).free / 1e9
print(f'\nDisk: used {disk_used:.1f} GB, free {disk_free:.1f} GB')
if disk_free < 15:
    print('⚠ Свободного места меньше 15 GB — следующая фаза (YOLO disk-cache + кропы) может не влезть.')


## 3. YOLO-датасет: один класс `leaf`

Детектор одноклассовый: только локализует листья, решение «болеет ли» делает классификатор.
PlantDoc даёт PASCAL VOC XML на 27 классов — все мапятся в `leaf`. Split 85/15 train/val.


In [ ]:
import xml.etree.ElementTree as ET
import random

YOLO_DIR = WORKDIR / 'detector'
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (YOLO_DIR / sub).mkdir(parents=True, exist_ok=True)

random.seed(42)
pairs: list[tuple[Path, Path]] = []
for split_name in ('TRAIN', 'TEST'):
    split_dir = plantdoc_dir / split_name
    if not split_dir.exists():
        continue
    for xml in split_dir.rglob('*.xml'):
        for ext in ('.jpg', '.JPG', '.jpeg', '.png'):
            img = xml.with_suffix(ext)
            if img.exists():
                pairs.append((img, xml))
                break
print(f'PlantDoc XML pairs: {len(pairs)}')
random.shuffle(pairs)
cut = int(len(pairs) * 0.85)
train_pairs, val_pairs = pairs[:cut], pairs[cut:]


def write_yolo(pair_list, split):
    kept = 0
    for img, xml in pair_list:
        try:
            root = ET.parse(xml).getroot()
        except ET.ParseError:
            continue
        size = root.find('size')
        if size is None:
            continue
        W = float(size.findtext('width') or 0)
        H = float(size.findtext('height') or 0)
        if W <= 0 or H <= 0:
            continue
        lines = []
        for obj in root.findall('object'):
            name = (obj.findtext('name') or '').lower()
            if 'leaf' not in name:
                continue
            bb = obj.find('bndbox')
            if bb is None:
                continue
            xmin = float(bb.findtext('xmin') or 0)
            ymin = float(bb.findtext('ymin') or 0)
            xmax = float(bb.findtext('xmax') or 0)
            ymax = float(bb.findtext('ymax') or 0)
            if xmax <= xmin or ymax <= ymin:
                continue
            cx = ((xmin + xmax) / 2) / W
            cy = ((ymin + ymax) / 2) / H
            bw = (xmax - xmin) / W
            bh = (ymax - ymin) / H
            cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
            bw, bh = min(max(bw, 0.001), 1), min(max(bh, 0.001), 1)
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        if not lines:
            continue
        _link_or_copy(img, YOLO_DIR / f'images/{split}' / img.name)
        (YOLO_DIR / f'labels/{split}' / (img.stem + '.txt')).write_text(
            '\n'.join(lines), encoding='utf-8')
        kept += 1
    return kept


n_tr = write_yolo(train_pairs, 'train')
n_vl = write_yolo(val_pairs, 'val')
print(f'YOLO train={n_tr}  val={n_vl}')

data_yaml = YOLO_DIR / 'data.yaml'
data_yaml.write_text(
    f'path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [leaf]\n',
    encoding='utf-8')
print('data.yaml →', data_yaml)


## 4. Тренируем YOLOv8n (Stage 1)

Одноклассовый детектор сходится быстрее. T4: ~70 мин на 60 эпох.
Ожидаемые финальные метрики: mAP50 ≈ 0.74-0.78, mAP50-95 ≈ 0.50.

- `cache='disk'` — экономит RAM Colab (~9.6 GB на cache vs RAM)
- `mosaic=1.0` + `close_mosaic=10` — устойчивость к шумным фото; финальные эпохи без mosaic
- `hsv_*` — устойчивость к индоор-освещению (тёплая лампа, окно)
- AMP + cos_lr + patience=12 — стандартный recipe

Артефакты сразу копируются на Drive (если смонтирован).


In [ ]:
from ultralytics import YOLO

det_model = YOLO('yolov8n.pt')

det_results = det_model.train(
    data=str(data_yaml),
    epochs=60,
    imgsz=640,
    batch=32,
    patience=12,
    cos_lr=True,
    amp=True,
    cache='disk',
    workers=4,
    save_period=-1,
    close_mosaic=10,
    # v4.6: mosaic=1.0 was over-strong for a single-class detector on
    # PlantDoc (already tightly cropped). 4-in-1 composites synthesise
    # tiny leaves that don't match the inference distribution
    # (typical phone shots have 1-3 large leaves). Dropping to 0.5 is
    # the standard recommendation for single-class fine-tuning.
    mosaic=0.5,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    project=str(WORKDIR / 'yolo_runs'),
    name='detector',
    exist_ok=True,
    verbose=True,
)

best_pt = WORKDIR / 'yolo_runs' / 'detector' / 'weights' / 'best.pt'
assert best_pt.exists(), f'YOLO best.pt не сохранён в {best_pt}'
MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)
shutil.copy2(best_pt, MODELS_OUT / 'detector.pt')
save_artifact(MODELS_OUT / 'detector.pt', 'detector PT after YOLO train')

det_model.export(format='onnx', imgsz=640, opset=13, simplify=True, dynamic=False)
onnx_src = best_pt.with_suffix('.onnx')
if onnx_src.exists():
    shutil.copy2(onnx_src, MODELS_OUT / 'detector.onnx')
    save_artifact(MODELS_OUT / 'detector.onnx', 'detector ONNX')
    print('Detector ONNX →', MODELS_OUT / 'detector.onnx')

metrics = det_model.val(data=str(data_yaml), imgsz=640, plots=False, verbose=False)
print(f'mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}')


## 5. Классификаторный датасет (Stage 2)

**Целевая таксономия v5.0 — 10 классов с graceful fallback:**

| Класс | Источник | План количества |
|-------|----------|----------------|
| `anthracnose`            | iNat *Colletotrichum* (323958) | 600-800 |
| `bacterial_leaf_spot`    | PV `Tomato___Bacterial_spot` + iNat *Pseudomonas syringae* (337405) → *Xanthomonas* (132056) | 600-800 |
| `gray_mold`              | iNat *Botrytis cinerea* (324263) → род → семейство | 600-800 |
| `healthy`                | Kaggle Houseplant Species через детектор (multi-crop+NMS) | 2500 |
| `iron_chlorosis`         | PV TYLCV (визуальный твин) | 500 |
| `late_blight`            | PV `Tomato/Potato___Late_blight` + iNat *Phytophthora* | 600-800 |
| `mosaic_virus`           | PV `Tomato___Tomato_mosaic_virus` + iNat *Tobamovirus* (916920) → *TMV* (1027410) | 600-800 |
| `not_a_plant`            | COCO val2017 минус plant-like | 1800 |
| `powdery_mildew`         | PV Squash/Cherry + iNat *Erysiphales* (55524) | 1100 |
| `sunburn`                | PV `Strawberry___Leaf_scorch` | 500 |

Защитный фильтр: класс с <60 train samples будет автоматически дропнут перед тренировкой, и итоговая модель станет (10-N)-class. Это значит, что любой неудавшийся iNat fetch не ломает пайплайн целиком.

**Ключевые санитайзеры v5.0:**
- **SHA-256 dedup** на (kind=file): любые байт-идентичные дубликаты между PV / iNat / COCO режутся.
- **EXIF strip** при materialize'е (Pillow `image.getdata()` → re-save без EXIF).
- **Magic-byte валидация** на скачивании уже была; на materialize дополнительно `Image.verify()`.
- **Group-aware split** сохраняется (iNat observation_id + multi-crop photo_id никогда не утекают между train/val).


In [ ]:
import json
import re
import hashlib
from PIL import Image
import numpy as np
import torch
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)

CLS_DIR = WORKDIR / 'classifier'
if CLS_DIR.exists():
    shutil.rmtree(CLS_DIR)

# Алфавитный порядок (как ImageFolder.classes отсортирует).
TARGET_CLASSES = [
    # PlantScope v5.0 — 10 disease-only classes (alphabetical).
    'anthracnose',
    'bacterial_leaf_spot',
    'gray_mold',
    'healthy',
    'iron_chlorosis',
    'late_blight',
    'mosaic_virus',
    'not_a_plant',
    'powdery_mildew',
    'sunburn',
]
random.seed(42)
VAL_RATIO = 0.15
MIN_VAL_PER_CLASS = 8

for split in ('train', 'val'):
    for cls in TARGET_CLASSES:
        (CLS_DIR / split / cls).mkdir(parents=True, exist_ok=True)

# v5.0: SHA-256 dedup across the whole pool. PV and iNat occasionally
# overlap (e.g. someone re-uploaded a PV photo as an iNat observation),
# and an undetected duplicate that lands in val while its sibling is in
# train inflates val accuracy. We hash file bytes for ('file', path)
# items and PIL bytes for ('pil', crop) items.
_SEEN_HASHES: set[str] = set()


def _file_hash(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _pil_hash(img: Image.Image) -> str:
    # Hash the raw normalised RGB bytes so different JPEG re-encodes of
    # the same crop still collapse to one hash.
    arr = np.asarray(img.convert('RGB').resize((128, 128), Image.BILINEAR), dtype=np.uint8)
    return hashlib.sha256(arr.tobytes()).hexdigest()


def _maybe_add(pool_list: list, item: tuple) -> bool:
    """Append item to pool_list iff its hash is fresh. item = (kind, payload, group)."""
    kind, payload = item[0], item[1]
    try:
        h = _file_hash(payload) if kind == 'file' else _pil_hash(payload)
    except Exception:
        return False
    if h in _SEEN_HASHES:
        return False
    _SEEN_HASHES.add(h)
    pool_list.append(item)
    return True


# Стратифицированный split: собираем все источники по классам, потом
# случайно делим каждый класс с фиксированной долей val.
# Каждый элемент: (kind, payload, group_key) где kind ∈ {'file', 'pil'}.
pool: dict[str, list[tuple]] = {c: [] for c in TARGET_CLASSES}


# ── 5.1 PlantVillage кропы ──
PV_TO_CLASS = {
    'powdery_mildew':       'powdery_mildew',
    'iron_chlorosis':       'iron_chlorosis',
    'late_blight':          'late_blight',
    'sunburn':              'sunburn',
    # v5.0 NEW
    'bacterial_leaf_spot':  'bacterial_leaf_spot',
    'mosaic_virus':         'mosaic_virus',
}
# v5.0 caps — slight rebalance: bacterial / mosaic get larger caps
# because PV is the dominant clean source for them, while iNat is
# noisier (community photos with hand/sky in the frame).
PV_CAP = {
    'powdery_mildew':       500,
    'iron_chlorosis':       500,
    'late_blight':          800,
    'sunburn':              500,
    'bacterial_leaf_spot':  700,
    'mosaic_virus':         700,
}
for pv_cls, dst_cls in PV_TO_CLASS.items():
    src_dir = pv_extracted / pv_cls
    if not src_dir.exists():
        continue
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    selected = imgs[:PV_CAP[pv_cls]]
    added = 0
    for img in selected:
        if _maybe_add(pool[dst_cls], ('file', img, None)):
            added += 1
    print(f'PV {pv_cls} → {dst_cls}: kept {added} / shuffled {len(selected)} (deduped)')

# ── 5.2 iNaturalist → disease classes only ──
# v5.0: pests are NOT classifier outputs. iNat now feeds 6 disease
# classes including the 2 new v5.0 buckets (bacterial / mosaic).
INAT_OTHER = {
    'powdery_mildew':       ('powdery_mildew',      600),
    'gray_mold':            ('gray_mold',           700),
    'late_blight':          ('late_blight',         400),
    'anthracnose':          ('anthracnose',         700),
    # v5.0 NEW — iNat is the primary source for these (PV exists but
    # is field-crop; iNat adds Araceae / Hoya / Dracaena variations).
    'bacterial_leaf_spot':  ('bacterial_leaf_spot', 500),
    'mosaic_virus':         ('mosaic_virus',        500),
}

def _inat_obs_group(path):
    # 't{taxon_id}_{obs_id}_{ph_id}' → obs_id (used to keep all photos
    # of the same observation in the SAME train/val split).
    m = re.match(r't(\d+)_(\d+)_\d+', path.stem)
    if m:
        return f'inat_obs_{m.group(2)}'
    return None


for key, (dst_cls, cap) in INAT_OTHER.items():
    src_dir = inat_root / key
    if not src_dir.exists():
        continue
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    selected = imgs[:cap]
    added = 0
    for img in selected:
        if _maybe_add(pool[dst_cls], ('file', img, _inat_obs_group(img))):
            added += 1
    print(f'iNat {key} → {dst_cls}: kept {added} / shuffled {len(selected)} (deduped)')


# ── 5.3 Houseplant Species → healthy: BATCHED multi-crop + NMS ──
HP_TARGET_HEALTHY = 2500
CROPS_PER_PHOTO = 3
DETECT_BATCH = 16


def _iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    a_area = (ax2 - ax1) * (ay2 - ay1)
    b_area = (bx2 - bx1) * (by2 - by1)
    return inter / (a_area + b_area - inter + 1e-9)


def _nms_keep(boxes, max_keep, iou_thr=0.5):
    boxes_sorted = sorted(boxes, key=lambda b: -((b[2]-b[0])*(b[3]-b[1])))
    kept = []
    for b in boxes_sorted:
        if any(_iou(b, k) > iou_thr for k in kept):
            continue
        kept.append(b)
        if len(kept) >= max_keep:
            break
    return kept


if hp_dir.exists():
    det = YOLO(str(MODELS_OUT / 'detector.pt'))
    hp_imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
        hp_imgs.extend(hp_dir.rglob(ext))
    random.shuffle(hp_imgs)
    saved_crop = 0
    healthy_added = 0

    def _open_safe(p):
        try:
            return Image.open(p).convert('RGB')
        except Exception:
            return None

    i = 0
    while i < len(hp_imgs) and healthy_added < HP_TARGET_HEALTHY:
        chunk = hp_imgs[i:i + DETECT_BATCH]
        i += DETECT_BATCH
        pils = [_open_safe(p) for p in chunk]
        valid_pairs = [(p, pi) for p, pi in zip(chunk, pils) if pi is not None]
        if not valid_pairs:
            continue
        arrs = [np.array(pi) for _, pi in valid_pairs]
        try:
            results = det.predict(arrs, imgsz=640, conf=0.4,
                                  verbose=False, device=device)
        except Exception:
            continue
        for (path, pil), res in zip(valid_pairs, results):
            if healthy_added >= HP_TARGET_HEALTHY:
                break
            if res.boxes is None or len(res.boxes) == 0:
                continue
            boxes = res.boxes.xyxy.cpu().numpy().tolist()
            H, W = pil.size[1], pil.size[0]
            kept = _nms_keep(boxes, CROPS_PER_PHOTO, iou_thr=0.5)
            photo_group = f'hp_{path.stem}'
            for b in kept:
                x1, y1, x2, y2 = (int(round(v)) for v in b)
                pad_x = int((x2 - x1) * 0.1)
                pad_y = int((y2 - y1) * 0.1)
                x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
                x2, y2 = min(W, x2 + pad_x), min(H, y2 + pad_y)
                if x2 - x1 < 32 or y2 - y1 < 32:
                    continue
                crop = pil.crop((x1, y1, x2, y2))
                if _maybe_add(pool['healthy'], ('pil', crop, photo_group)):
                    saved_crop += 1
                    healthy_added += 1
                if healthy_added >= HP_TARGET_HEALTHY:
                    break
    print(f'Houseplant Species → healthy: {saved_crop} crops via detector (batched multi-crop, deduped)')

    # КРИТИЧНО: освобождаем VRAM от YOLO до старта classifier-фазы.
    del det
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# ── 5.4 COCO → not_a_plant ──
PLANT_LIKE_COCO_IDS = {
    64,  # potted plant
    86,  # vase
    56,  # broccoli
    50, 51, 52, 53, 54, 55, 57, 58, 59, 60, 61,
}
banned: set[int] = set()
if coco_ann.exists():
    coco_meta = json.loads(coco_ann.read_text(encoding='utf-8'))
    for a in coco_meta.get('annotations', []):
        if a.get('category_id') in PLANT_LIKE_COCO_IDS:
            banned.add(a['image_id'])
    print(f'COCO ban-list: {len(banned)} images excluded')
    del coco_meta
    gc.collect()


def _cid(p: Path) -> int:
    try:
        return int(p.stem)
    except ValueError:
        return -1


coco_imgs = [p for p in coco_dir.glob('*.jpg') if _cid(p) not in banned]
random.shuffle(coco_imgs)
# v5.0: hold COCO_CAP at 1800 (v4.7 bump, validated). With 10 classes
# instead of 8, NAP no longer dominates the WeightedRandomSampler.
COCO_CAP = 1800
nap_added = 0
for img in coco_imgs[:COCO_CAP]:
    if _maybe_add(pool['not_a_plant'], ('file', img, None)):
        nap_added += 1
print(f'COCO → not_a_plant: {nap_added}')


# ── 5.6 Стратифицированный split + EXIF-strip materialize ──
stats = {c: {'train': 0, 'val': 0} for c in TARGET_CLASSES}


def _strip_and_save(img: Image.Image, dst: Path) -> bool:
    """v5.0: re-encode without EXIF / metadata (privacy + size).

    PIL ``Image.save`` includes EXIF by default when given a JPEG with
    one. We rebuild the pixel data from a bare ``Image.new`` + paste
    so the saved file has only the raw RGB pixels — no GPS, camera
    serial or thumbnail.
    """
    try:
        clean = Image.new('RGB', img.size)
        clean.paste(img.convert('RGB'))
        clean.save(dst, 'JPEG', quality=92, optimize=True)
        return True
    except Exception:
        return False


def _materialize(items: list[tuple], cls: str, split: str):
    out_dir = CLS_DIR / split / cls
    for item in items:
        kind, payload = item[0], item[1]
        idx = stats[cls][split]
        if kind == 'file':
            ext = payload.suffix.lower() or '.jpg'
            # v5.0: open + EXIF-strip + JPEG re-encode for ALL items so
            # the val/train materialised files don't carry EXIF or
            # corrupted JPEGs through to ImageFolder. ~1.5x extra
            # disk I/O vs hardlink, but we get verified bytes.
            try:
                with Image.open(payload) as src:
                    src.load()
                    dst = out_dir / f'{cls}_{idx:06d}.jpg'
                    if _strip_and_save(src, dst):
                        stats[cls][split] += 1
            except Exception:
                pass
        else:
            dst = out_dir / f'{cls}_{idx:06d}.jpg'
            if _strip_and_save(payload, dst):
                stats[cls][split] += 1


def _group_aware_split(items, val_ratio, min_val):
    """Stratified split that keeps grouped items (same iNat
    observation, multi-crop from same photo) on the SAME side of the
    split. Items without a group are treated as their own singleton.
    """
    groups: dict = {}
    for i, item in enumerate(items):
        key = item[2] if (len(item) >= 3 and item[2] is not None) else f'_solo_{i}'
        groups.setdefault(key, []).append(item)
    keys = list(groups.keys())
    random.shuffle(keys)
    n_total = len(items)
    target_val = max(int(round(n_total * val_ratio)),
                     min_val if n_total >= min_val * 2 else 1)
    target_val = min(target_val, n_total - 1)
    val_items, train_items = [], []
    for key in keys:
        bucket = val_items if len(val_items) < target_val else train_items
        bucket.extend(groups[key])
    return train_items, val_items


for cls, items in pool.items():
    if not items:
        continue
    train_items, val_items = _group_aware_split(items, VAL_RATIO, MIN_VAL_PER_CLASS)
    _materialize(train_items, cls, 'train')
    _materialize(val_items, cls, 'val')

# Освобождаем pool (PIL crops держали reference на whole-image bytes).
del pool
gc.collect()

# ── 5.7 Защитный фильтр (<60 train) ──
MIN_TRAIN_PER_CLASS = 60
dropped: list[str] = []
for cls in list(TARGET_CLASSES):
    if stats[cls]['train'] < MIN_TRAIN_PER_CLASS:
        dropped.append(cls)
        for split in ('train', 'val'):
            d = CLS_DIR / split / cls
            if d.exists():
                shutil.rmtree(d)
        TARGET_CLASSES.remove(cls)
        stats.pop(cls, None)

print('\nРаспределение по классам (стратифицированный split, после dedup + EXIF strip):')
for cls in TARGET_CLASSES:
    print(f'  {cls:<24}  train={stats[cls]["train"]:>4}  val={stats[cls]["val"]:>4}')
total_train = sum(s["train"] for s in stats.values())
total_val   = sum(s["val"]   for s in stats.values())
print(f'\nИтого: {total_train} train / {total_val} val  ({len(TARGET_CLASSES)} классов)')
print(f'Уникальных хешей: {len(_SEEN_HASHES)} (дубликатов отсечено: см. отдельные kept/shuffled выше)')
if dropped:
    print('\n⚠ Отсечены (<60 train):', dropped)
    print('  Это значит, что соответствующий iNat-источник вернул мало фото.')
    print(f'  Финальная модель будет {len(TARGET_CLASSES)}-class вместо 10.')

# v5.0: balance sanity check. If one disease class has >5× the
# samples of another disease, the WeightedRandomSampler upweights
# the small class so aggressively that mixup partners are almost
# always the dominant class — a subtle leak. Warn so the user can
# rebalance the caps in PV_CAP / INAT_OTHER if needed.
disease_counts = [
    (cls, stats[cls]['train'])
    for cls in TARGET_CLASSES if cls not in ('healthy', 'not_a_plant')
]
if disease_counts:
    mn = min(c for _, c in disease_counts)
    mx = max(c for _, c in disease_counts)
    if mx > 5 * max(mn, 1):
        ratio = mx / max(mn, 1)
        print(f'⚠ Разрыв между болезнями x{ratio:.1f} (max={mx}, min={mn}). '
              'Подумай о rebalance в PV_CAP / INAT_OTHER.')


## 6. EfficientNet-B0 — обучение классификатора (v4.7)

| Аспект | Значение | Обоснование |
|--------|----------|-------------|
| Backbone        | **EfficientNet-B0** (~5M, 240px) | torchvision IMAGENET1K_V1. v4.7 откатился с V2-S (21M @ 320) на B0 потому что (a) датасет ~7-8k картинок — V2-S переразмерен и склонен к overfitting; (b) сервер крутится на CPU-only VPS — B0 в ~4× быстрее; (c) ONNX INT8 классификатор после экспорта весит ~5MB вместо ~85MB |
| Resolution      | 240 px | сладкая точка между native 224 (быстрее) и 320 (детальнее) |
| Loss            | CrossEntropy + label_smoothing **0.05** | в v4.3 был 0.1 → underconfident (T=0.5) |
| Балансировка    | WeightedRandomSampler | без двойной балансировки (sampler+weights) |
| CutMix          | p=**0.4** (только plant↔plant) | NAP никогда не мешается с plant-classes |
| MixUp           | p=**0.2** (NAP-safe) | v4.7: доп. регуляризатор поверх CutMix |
| TTA             | h-flip + center-crop | |
| EMA decay       | 0.999 | calibrated для 22-эпохного трейна |
| Grad clip       | 1.0 | после `scaler.unscale_` |
| Optimizer       | AdamW + warmup + CosineAnnealing | |
| Augmentation (v4.6+) | TrivialAugmentWide + WarmLight + JPEG + RandomErasing + MotionBlur + Shadow | |

Phase A (head only, 8 ep) → Phase B (fine-tune features.5-8 + classifier, 14 ep). T4: ~30-40 мин.

**Защита от OOM**: streaming val pass без накопления logits, explicit `gc.collect()` между фазами.

**Защита от переобучения**: на ~7-8k датасете B0 (5M) имеет лучший overfit-to-capacity баланс чем V2-S (21M) + aug + label_smoothing + cutmix + mixup + EMA + early-stop.

**Реальные фото**: WarmLight, JPEG, MotionBlur, RandomShadow.

**OOD / not_a_plant**: в production pipeline стоит MSP-gate (`pipeline.py:NAP_*_FLOOR`), который маршрутизирует низко-уверенные ответы в `not_a_plant` без guess'a болезни.

In [ ]:
import copy, io, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
)
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import classification_report, confusion_matrix

# ── v5.0 training flags (toggle at the top of the cell) ──
TRAIN_ENSEMBLE_PEER = True   # train EfficientNetV2-S @ 300 as a second
                             # checkpoint and export it as classifier_v2s*.onnx.
                             # Server will weighted-average its softmax with B0.
USE_FOCAL_LOSS      = False  # γ=2.0 focal CE. Helps when not_a_plant /
                             # healthy dominate gradients; default off
                             # because the WeightedRandomSampler already
                             # rebalances and label_smoothing CE is the
                             # better-calibrated baseline.
DETERMINISTIC       = False  # set True for bit-reproducible runs
                             # (slower; disables some cuDNN heuristics).
GRAD_CHECKPOINTING_PEER = False  # turn on if V2-S OOMs on T4. ~30%
                                 # slower but ~40% less peak VRAM.
USE_TORCH_COMPILE   = False  # opt-in: torch.compile(net) on PyTorch
                             # 2.x. ~5-15% throughput gain on Ampere+
                             # but adds ~30-60s warmup and is unstable
                             # on T4 — leave off for the default recipe.
RESUME_FROM_BACKUP  = True   # if BACKUP_DIR (PLANTSCOPE_BACKUP_DIR)
                             # contains a prior best.pth or
                             # classifier.pth, warm-start B0 from it.
LABEL_NOISE_LOG     = True   # log val examples where the model is
                             # >95% sure on a different class than the
                             # ImageFolder label — surfaces mislabels.

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if DETERMINISTIC:
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Free GPU memory left over from earlier cells (notably the YOLO det
# loaded in cell 10 for the houseplant crop pass).
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# v5.0 hyperparameters (B0 @ 240 primary + optional V2-S @ 300 peer).
IMG_SIZE = 240
PEER_IMG_SIZE = 300
BATCH_SIZE = 96
PEER_BATCH_SIZE = 48     # V2-S @ 300 — ~11 GB VRAM on T4, batch=48 fits.
NUM_WORKERS = min(os.cpu_count() or 2, 4)
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.05
CUTMIX_PROB = 0.4
MIXUP_PROB = 0.2
MIXUP_ALPHA = 0.2
EMA_DECAY = 0.999
EARLY_STOP_PATIENCE = 5
FOCAL_GAMMA = 2.0
PEER_WEIGHT_DEFAULT = 0.45  # softmax-blend weight for the V2-S peer
                            # (persisted into classes.json["ensemble"]).


# ── Custom augmentations for indoor-photo robustness ──

class RandomJPEGCompression:
    """Симулирует JPEG-артефакты пользовательских фото."""
    def __init__(self, p=0.5, q_range=(40, 95)):
        self.p, self.q_min, self.q_max = p, q_range[0], q_range[1]
    def __call__(self, img):
        if random.random() > self.p:
            return img
        q = random.randint(self.q_min, self.q_max)
        buf = io.BytesIO()
        img.convert('RGB').save(buf, format='JPEG', quality=q)
        buf.seek(0)
        out = Image.open(buf)
        out.load()
        return out.convert('RGB')


class WarmLightSimulation:
    """Тёплая индоор-лампочка: warm tint + low contrast (PIL native)."""
    def __init__(self, p=0.3):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if img.mode != 'RGB':
            img = img.convert('RGB')
        r, g, b = img.split()
        r_factor = 1.05 + random.random() * 0.10
        b_factor = 0.85 - random.random() * 0.10
        r = r.point(lambda v, k=r_factor: min(255, int(v * k)))
        b = b.point(lambda v, k=b_factor: int(v * k))
        img = Image.merge('RGB', (r, g, b))
        gamma = 0.85 + random.random() * 0.15
        img = ImageEnhance.Contrast(img).enhance(gamma)
        return img


class MotionBlur:
    """Симулирует дрожание руки при съёмке."""
    def __init__(self, p=0.15, max_kernel=5):
        self.p = p
        self.max_kernel = max_kernel
    def __call__(self, img):
        if random.random() > self.p:
            return img
        radius = random.uniform(0.5, 1.5)
        return img.filter(ImageFilter.BoxBlur(radius))


class RandomShadow:
    """Боковая тень от окна. Линейный градиент яркости."""
    def __init__(self, p=0.2, intensity=(0.5, 0.85)):
        self.p = p
        self.intensity = intensity
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if img.mode != 'RGB':
            img = img.convert('RGB')
        w, h = img.size
        side = random.choice(['left', 'right', 'top', 'bottom'])
        intensity = random.uniform(*self.intensity)
        mask_arr = np.ones((h, w), dtype=np.float32)
        if side == 'left':
            grad = np.linspace(intensity, 1.0, w)[None, :]
            mask_arr = np.tile(grad, (h, 1))
        elif side == 'right':
            grad = np.linspace(1.0, intensity, w)[None, :]
            mask_arr = np.tile(grad, (h, 1))
        elif side == 'top':
            grad = np.linspace(intensity, 1.0, h)[:, None]
            mask_arr = np.tile(grad, (1, w))
        else:
            grad = np.linspace(1.0, intensity, h)[:, None]
            mask_arr = np.tile(grad, (1, w))
        arr = np.array(img, dtype=np.float32)
        arr = arr * mask_arr[..., None]
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        return Image.fromarray(arr, 'RGB')


def make_train_tf(size: int) -> transforms.Compose:
    """v5.0: aug stack parametrised by image size so peer training (300px)
    reuses the same logic at a different resolution."""
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.1),
        WarmLightSimulation(p=0.3),
        RandomShadow(p=0.20),
        MotionBlur(p=0.15),
        RandomJPEGCompression(p=0.5, q_range=(50, 95)),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])


def make_val_tf(size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


# ── Datasets / loaders (B0 path) ──
train_ds = datasets.ImageFolder(str(CLS_DIR / 'train'), transform=make_train_tf(IMG_SIZE))
val_ds   = datasets.ImageFolder(str(CLS_DIR / 'val'),   transform=make_val_tf(IMG_SIZE))
class_names = list(train_ds.classes)
num_classes = len(class_names)
not_a_plant_idx = class_names.index('not_a_plant') if 'not_a_plant' in class_names else -1
healthy_idx = class_names.index('healthy') if 'healthy' in class_names else -1
print('Classes:', class_names)
print('not_a_plant idx =', not_a_plant_idx, '| healthy idx =', healthy_idx)

# v5.0 balance assertion. If the smallest class < 0.5% of total, the
# WeightedRandomSampler can swing wildly; warn so the user can rebuild
# pool with bigger caps before throwing GPU hours at it.
counts = [0] * num_classes
for _, y in train_ds.samples:
    counts[y] += 1
total = sum(counts)
# v5.0 perf: torch.DoubleTensor sample weights — Python-list → tensor
# saves ~5-10ms / epoch on 10k samples and avoids the per-call list
# materialisation inside WeightedRandomSampler.
weights_t = torch.tensor(
    [total / (counts[y] or 1) for _, y in train_ds.samples],
    dtype=torch.double,
)
sample_w = weights_t  # alias kept for legacy refs
sampler = WeightedRandomSampler(weights_t, num_samples=len(weights_t), replacement=True)

if total > 0:
    minc = min(counts) if counts else 0
    if minc < 60:
        print(f'⚠ Smallest class has only {minc} train samples; consider increasing iNat caps.')

pin = device.type == 'cuda'

def make_loaders(train_ds_, val_ds_, batch_size, pin_):
    # v5.0 perf: prefetch_factor=4 keeps the GPU fed even when
    # workers stall on a slow JPEG (Pillow's BILINEAR resize on a
    # 4k photo is the dominant cost). 4× memory budget per worker
    # but well under T4's 13 GB host RAM cap at batch≤96.
    pf = 4 if NUM_WORKERS > 0 else None
    tl = DataLoader(
        train_ds_, batch_size=batch_size, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=pin_,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=pf,
        drop_last=True,
    )
    vl = DataLoader(
        val_ds_, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin_,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=pf,
    )
    return tl, vl


train_loader, val_loader = make_loaders(train_ds, val_ds, BATCH_SIZE, pin)


def build_b0_net(num_classes_: int) -> nn.Module:
    net_ = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    net_.classifier[-1] = nn.Linear(net_.classifier[-1].in_features, num_classes_)
    for name, p in net_.named_parameters():
        p.requires_grad = name.startswith('classifier.')
    return net_.to(memory_format=torch.channels_last).to(device)


def build_v2s_net(num_classes_: int) -> nn.Module:
    net_ = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    net_.classifier[-1] = nn.Linear(net_.classifier[-1].in_features, num_classes_)
    if GRAD_CHECKPOINTING_PEER:
        # Wrap the heavy stages in checkpoint_sequential at training
        # time. Inference stays full-precision (peer is exported in eval
        # mode after Phase B), so this only affects memory/throughput
        # during fine-tuning.
        try:
            from torch.utils.checkpoint import checkpoint_sequential  # noqa: WPS433
            features = net_.features
            net_._forward_features = features  # keep ref for export
            def fwd(x):
                x = checkpoint_sequential(features, len(features), x, use_reentrant=False)
                x = net_.avgpool(x)
                x = torch.flatten(x, 1)
                return net_.classifier(x)
            net_.forward = fwd  # type: ignore
        except Exception as e:
            print(f'  ! Gradient checkpointing not active: {e}')
    for name, p in net_.named_parameters():
        p.requires_grad = name.startswith('classifier.')
    return net_.to(memory_format=torch.channels_last).to(device)


net = build_b0_net(num_classes)


# ── Optional warm-start from Drive ──
def _maybe_resume_b0(net_: nn.Module) -> bool:
    if not RESUME_FROM_BACKUP:
        return False
    candidates: list[Path] = []
    # Local prior run takes precedence over backup.
    for name in ('best.pth', 'classifier.pth'):
        local = MODELS_OUT / name
        if local.exists():
            candidates.append(local)
    # Backup dir (env-driven) — any prior run mirrored there.
    if BACKUP_DIR is not None:
        for name in ('best.pth', 'classifier.pth'):
            p = BACKUP_DIR / name
            if p.exists():
                candidates.append(p)
    for p in candidates:
        if p.exists():
            try:
                state = torch.load(str(p), map_location='cpu', weights_only=True)
                # Head shape may differ across versions (8 vs 10 classes).
                # Drop the last classifier layer if shapes don't match.
                head_keys = [k for k in state if k.startswith('classifier.') and k.endswith('.weight') and state[k].ndim == 2]
                for hk in head_keys:
                    if state[hk].shape[0] != num_classes:
                        # Skip the entire classifier — Phase A will retrain
                        # the head from scratch.
                        for k in list(state.keys()):
                            if k.startswith('classifier.'):
                                del state[k]
                        print(f'  · Drive checkpoint head mismatch — resuming features only.')
                        break
                missing, unexpected = net_.load_state_dict(state, strict=False)
                print(f'  ✓ Warm-start from {p.name} (missing={len(missing)}, unexpected={len(unexpected)})')
                return True
            except Exception as e:
                print(f'  ! resume failed from {p}: {e}')
    return False


_maybe_resume_b0(net)

if USE_TORCH_COMPILE and hasattr(torch, 'compile') and device.type == 'cuda':
    try:
        net = torch.compile(net, mode='reduce-overhead', dynamic=False)
        print('  ✓ torch.compile enabled (mode=reduce-overhead).')
    except Exception as e:
        print(f'  ! torch.compile failed, continuing eager: {e}')


# ── CutMix / MixUp helpers ──

def _rand_bbox(w, h, lam):
    cut_rat = math.sqrt(max(0.0, 1.0 - lam))
    cw = int(w * cut_rat); ch = int(h * cut_rat)
    cx = random.randint(0, w - 1); cy = random.randint(0, h - 1)
    return (max(0, cx - cw // 2), max(0, cy - ch // 2),
            min(w, cx + cw // 2), min(h, cy + ch // 2))


def _safe_perm(y):
    """Перестановка с сохранением not_a_plant — не миксим NAP с растениями."""
    if not_a_plant_idx < 0:
        return torch.randperm(y.size(0), device=y.device)
    is_nap = (y == not_a_plant_idx)
    perm = torch.arange(y.size(0), device=y.device)
    plant_idx = torch.where(~is_nap)[0]
    nap_idx = torch.where(is_nap)[0]
    if len(plant_idx) > 1:
        perm[plant_idx] = plant_idx[torch.randperm(len(plant_idx), device=y.device)]
    if len(nap_idx) > 1:
        perm[nap_idx] = nap_idx[torch.randperm(len(nap_idx), device=y.device)]
    return perm


def cutmix_batch(x, y, alpha=1.0):
    lam = float(np.random.beta(alpha, alpha))
    perm = _safe_perm(y)
    bx1, by1, bx2, by2 = _rand_bbox(x.size(3), x.size(2), lam)
    x[:, :, by1:by2, bx1:bx2] = x[perm, :, by1:by2, bx1:bx2]
    lam = 1.0 - ((bx2 - bx1) * (by2 - by1)) / (x.size(3) * x.size(2))
    return x, y, y[perm], lam


def mixup_batch(x, y, alpha=MIXUP_ALPHA):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    perm = _safe_perm(y)
    x = lam * x + (1.0 - lam) * x[perm]
    return x, y, y[perm], lam


# ── Loss (CE label-smoothed OR focal CE behind a flag) ──

class FocalCE(nn.Module):
    """v5.0 optional: focal CE with label smoothing.

    Equivalent to CE when γ=0; γ>0 down-weights well-classified
    samples so the head spends more capacity on hard cases (e.g.
    bacterial vs anthracnose, healthy vs early mosaic).
    """
    def __init__(self, gamma: float = 2.0, label_smoothing: float = 0.05):
        super().__init__()
        self.gamma = gamma
        self.ls = label_smoothing
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=-1)
        if self.ls > 0:
            n_cls = logits.size(-1)
            with torch.no_grad():
                soft = torch.full_like(logp, self.ls / n_cls)
                soft.scatter_(1, target.view(-1, 1), 1.0 - self.ls + self.ls / n_cls)
            ce = -(soft * logp).sum(dim=-1)
        else:
            ce = F.nll_loss(logp, target, reduction='none')
        with torch.no_grad():
            pt = logp.gather(1, target.view(-1, 1)).exp().squeeze(1)
        focal = (1 - pt).clamp(min=1e-8) ** self.gamma
        return (focal * ce).mean()


crit = FocalCE(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING) if USE_FOCAL_LOSS \
       else nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print(f'Loss: {"FocalCE γ=" + str(FOCAL_GAMMA) if USE_FOCAL_LOSS else "CE label_smoothing=" + str(LABEL_SMOOTHING)}')

use_amp = device.type == 'cuda'
try:
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    autocast_ctx = lambda: torch.amp.autocast('cuda', enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    autocast_ctx = lambda: torch.cuda.amp.autocast(enabled=use_amp)


class ModelEMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for ep, p in zip(self.ema.parameters(), model.parameters()):
            ep.mul_(d).add_(p.detach(), alpha=1.0 - d)
        for eb, b in zip(self.ema.buffers(), model.buffers()):
            eb.copy_(b)


ema = ModelEMA(net, decay=EMA_DECAY)
MODELS_OUT = WORKDIR / 'out'; MODELS_OUT.mkdir(exist_ok=True)


def make_opt_sched(params, lr, epochs):
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    warm = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=1)
    cos = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - 1, 1))
    return opt, optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[1])


def _tta_predict(model_to_eval, x):
    """h-flip TTA. 5-crop is added in the final pass cell (post-train)."""
    with autocast_ctx():
        p1 = F.softmax(model_to_eval(x), dim=1)
        p2 = F.softmax(model_to_eval(torch.flip(x, dims=[3])), dim=1)
    return (p1 + p2) * 0.5


# ── Train / val loop ──

def run_train_epoch(loader, optimizer, model, ema_, criterion):
    model.train()
    loss_sum = correct = total_n = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        r = random.random()
        use_cutmix = r < CUTMIX_PROB
        use_mixup = (not use_cutmix) and r < CUTMIX_PROB + MIXUP_PROB
        use_mix = use_cutmix or use_mixup
        if use_cutmix:
            x, y_a, y_b, lam = cutmix_batch(x, y, alpha=1.0)
        elif use_mixup:
            x, y_a, y_b, lam = mixup_batch(x, y, alpha=MIXUP_ALPHA)
        with autocast_ctx():
            logits = model(x)
            if use_mix:
                loss = lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)
            else:
                loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            (p for p in model.parameters() if p.requires_grad), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ema_.update(model)
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total_n += x.size(0)
    return loss_sum / total_n, correct / total_n


@torch.inference_mode()
def run_val_epoch(model_to_eval, loader_):
    model_to_eval.eval()
    loss_sum = correct = total_n = 0
    for x, y in loader_:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        probs = _tta_predict(model_to_eval, x)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-8)), y)
        loss_sum += loss.item() * x.size(0)
        correct += (probs.argmax(1) == y).sum().item()
        total_n += x.size(0)
    return loss_sum / total_n, correct / total_n


best_acc = 0.0
best_state = {k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()}
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
no_improve_streak = 0
training_log_path = MODELS_OUT / 'training_log.txt'


def _log_epoch(line: str):
    print(line)
    try:
        with training_log_path.open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass


def phase(lbl, epochs, lr, allow_early_stop, model, ema_, criterion, train_loader_, val_loader_):
    global best_acc, best_state, no_improve_streak
    opt, sch = make_opt_sched([p for p in model.parameters() if p.requires_grad], lr, epochs)
    _log_epoch(f'\n=== {lbl} ===')
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = run_train_epoch(train_loader_, opt, model, ema_, criterion)
        vl_l, vl_a = run_val_epoch(ema_.ema, val_loader_)
        sch.step()
        history['train_loss'].append(tr_l); history['train_acc'].append(tr_a)
        history['val_loss'].append(vl_l);   history['val_acc'].append(vl_a)
        line = (f'  {ep}/{epochs}  train_loss={tr_l:.3f} acc={tr_a:.3f}  '
                f'val_loss={vl_l:.3f} acc={vl_a:.3f}  '
                f'lr={opt.param_groups[0]["lr"]:.2e}  ({time.time()-t0:.1f}s)')
        _log_epoch(line)
        if vl_a > best_acc + 1e-4:
            best_acc = vl_a
            best_state = {k: v.detach().cpu().clone()
                          for k, v in ema_.ema.state_dict().items()}
            torch.save(best_state, MODELS_OUT / 'best.pth')
            save_artifact(MODELS_OUT / 'best.pth', f'best.pth (val_acc={vl_a:.3f})')
            no_improve_streak = 0
        else:
            no_improve_streak += 1
        if allow_early_stop and no_improve_streak >= EARLY_STOP_PATIENCE:
            _log_epoch(f'  ⏸ Early stop: no improvement for {EARLY_STOP_PATIENCE} epochs')
            break


# ── B0 Phase A (head only) ──
phase('Phase A — B0 head only', epochs=8, lr=1e-3,
      allow_early_stop=False, model=net, ema_=ema, criterion=crit,
      train_loader_=train_loader, val_loader_=val_loader)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── B0 Phase B (fine-tune blocks 5-8 + classifier) ──
for name, p in net.named_parameters():
    p.requires_grad = (
        name.startswith('features.5') or
        name.startswith('features.6') or
        name.startswith('features.7') or
        name.startswith('features.8') or
        name.startswith('classifier')
    )
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'\nPhase B — fine-tune ({trainable:,} trainable params)')

no_improve_streak = 0
phase('Phase B — B0 fine-tune', epochs=14, lr=5e-5,
      allow_early_stop=True, model=net, ema_=ema, criterion=crit,
      train_loader_=train_loader, val_loader_=val_loader)

ema.ema.load_state_dict(best_state)
net.load_state_dict(best_state)
print(f'\nB0 Best val accuracy: {best_acc:.4f}  ({best_acc*100:.1f}%)')

torch.save(best_state, MODELS_OUT / 'classifier.pth')
save_artifact(MODELS_OUT / 'classifier.pth', 'classifier.pth (final B0)')
b0_best_acc = best_acc


# ── Final val pass for B0 (streaming) ──
print('\n── Final val pass B0 (streaming, logits + TTA preds) ──')
ema.ema.eval()
all_logits_cpu, all_y_cpu, all_pred_tta_cpu = [], [], []
with torch.inference_mode():
    for x, y in val_loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast_ctx():
            logits = ema.ema(x)
        probs_tta = _tta_predict(ema.ema, x)
        all_logits_cpu.append(logits.float().cpu())
        all_pred_tta_cpu.append(probs_tta.argmax(1).cpu())
        all_y_cpu.append(y)
        del logits, probs_tta, x

all_logits = torch.cat(all_logits_cpu)
all_y = torch.cat(all_y_cpu)
all_pred_tta = torch.cat(all_pred_tta_cpu)
del all_logits_cpu, all_pred_tta_cpu, all_y_cpu

# ── Temperature calibration ──
print('\n── Temperature calibration ──')
Ts = np.concatenate([
    np.linspace(0.3, 1.0, 36),
    np.linspace(1.05, 5.0, 80),
])
best_T, best_nll = 1.0, float('inf')
for T in Ts:
    nll = F.cross_entropy(all_logits / T, all_y).item()
    if nll < best_nll:
        best_nll = nll
        best_T = float(T)
raw_nll = F.cross_entropy(all_logits, all_y).item()
print(f'Optimal T = {best_T:.3f}  (NLL: {raw_nll:.4f} → {best_nll:.4f})')
if best_T <= 0.32:
    print('  ⚠ T уперлось в нижнюю границу. Регуляризация всё ещё избыточна.')
elif best_T >= 4.9:
    print('  ⚠ T уперлось в верхнюю границу. Модель overconfident, возможно overfitting.')
elif 0.85 <= best_T <= 1.5:
    print('  ✓ T в нативном диапазоне — модель хорошо откалибрована.')

(MODELS_OUT / 'temperature.json').write_text(
    json.dumps({
        'temperature': best_T,
        'val_nll_raw': round(raw_nll, 4),
        'val_nll_calibrated': round(best_nll, 4),
        '_comment': 'Divide logits by `temperature` before softmax for calibrated probabilities.',
    }, indent=2),
    encoding='utf-8',
)
save_artifact(MODELS_OUT / 'temperature.json', 'temperature.json')

# ── Per-class metrics ──
print('\n── Per-class metrics (val, TTA h-flip) ──')
all_pred_list = all_pred_tta.tolist()
all_true_list = all_y.tolist()
print(classification_report(all_true_list, all_pred_list,
                            target_names=class_names, digits=3, zero_division=0))
cm = confusion_matrix(all_true_list, all_pred_list, labels=list(range(num_classes)))
print('Confusion matrix:')
print('              ' + ' '.join(f'{n[:10]:>11}' for n in class_names))
for name, row in zip(class_names, cm, strict=False):
    print(f'{name:>14}  ' + ' '.join(f'{v:>11}' for v in row))

# v5.0 label-noise log: surface val examples where the model is
# >95% confident on a different class than the ImageFolder label.
# Most of those are dataset mislabels (PV proxy class boundaries are
# fuzzy) — a manual review pass usually trims 1-2% of training data
# and gains ~0.5% top-1.
if LABEL_NOISE_LOG:
    suspicious = []
    val_paths = [p for p, _ in val_ds.samples]
    soft = F.softmax(all_logits / max(best_T, 1e-3), dim=1)
    conf_max, pred = soft.max(dim=1)
    for i in range(len(all_y)):
        if pred[i].item() != all_y[i].item() and conf_max[i].item() > 0.95:
            suspicious.append((val_paths[i], class_names[all_y[i].item()],
                               class_names[pred[i].item()], float(conf_max[i].item())))
    if suspicious:
        print(f'\n── Label-noise candidates ({len(suspicious)} val items > 95% confident on a different class) ──')
        for p, true, pr, c in suspicious[:25]:
            print(f'  {Path(p).name:<40}  label={true:<22}  pred={pr:<22}  conf={c:.3f}')
        if len(suspicious) > 25:
            print(f'  ... and {len(suspicious) - 25} more')

# Free RAM/VRAM before peer training (V2-S phase) and ONNX export.
del all_logits, all_y, all_pred_tta
del train_loader  # release dataloader workers
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n✓ B0 phases complete. Memory cleared.')


# ── Optional ensemble peer (V2-S @ 300) ──
peer_best_acc = None
peer_best_state = None
if TRAIN_ENSEMBLE_PEER:
    print('\n══════════════════════════════════════════════════')
    print('  Training ensemble peer: EfficientNetV2-S @ 300 ')
    print('══════════════════════════════════════════════════')

    # Re-make datasets at the peer resolution. ImageFolder is cheap to
    # rebuild; we only need to swap the transforms.
    peer_train_ds = datasets.ImageFolder(str(CLS_DIR / 'train'), transform=make_train_tf(PEER_IMG_SIZE))
    peer_val_ds   = datasets.ImageFolder(str(CLS_DIR / 'val'),   transform=make_val_tf(PEER_IMG_SIZE))
    assert list(peer_train_ds.classes) == class_names, 'Peer class order must match B0.'
    peer_sampler = WeightedRandomSampler(
        sample_w, num_samples=len(sample_w), replacement=True,
    )
    _pf = 4 if NUM_WORKERS > 0 else None
    peer_train_loader = DataLoader(
        peer_train_ds, batch_size=PEER_BATCH_SIZE, sampler=peer_sampler,
        num_workers=NUM_WORKERS, pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=_pf,
        drop_last=True,
    )
    peer_val_loader = DataLoader(
        peer_val_ds, batch_size=PEER_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=_pf,
    )

    peer_net = build_v2s_net(num_classes)
    peer_ema = ModelEMA(peer_net, decay=EMA_DECAY)
    peer_crit = FocalCE(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING) if USE_FOCAL_LOSS \
                else nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    # Stash B0 best so we can restore the global "best" after peer phase.
    saved_best_acc = best_acc
    saved_best_state = {k: v.clone() for k, v in best_state.items()}
    best_acc = 0.0
    best_state = {k: v.detach().cpu().clone() for k, v in peer_ema.ema.state_dict().items()}
    no_improve_streak = 0

    phase('Peer Phase A — V2-S head only', epochs=6, lr=1e-3,
          allow_early_stop=False, model=peer_net, ema_=peer_ema,
          criterion=peer_crit,
          train_loader_=peer_train_loader, val_loader_=peer_val_loader)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # V2-S has 8 feature stages (0..7); unfreeze the last few + classifier.
    for name, p in peer_net.named_parameters():
        p.requires_grad = (
            name.startswith('features.4') or
            name.startswith('features.5') or
            name.startswith('features.6') or
            name.startswith('features.7') or
            name.startswith('classifier')
        )

    no_improve_streak = 0
    phase('Peer Phase B — V2-S fine-tune', epochs=10, lr=4e-5,
          allow_early_stop=True, model=peer_net, ema_=peer_ema,
          criterion=peer_crit,
          train_loader_=peer_train_loader, val_loader_=peer_val_loader)

    peer_best_acc = best_acc
    peer_best_state = {k: v.clone() for k, v in best_state.items()}
    peer_ema.ema.load_state_dict(peer_best_state)
    peer_net.load_state_dict(peer_best_state)
    torch.save(peer_best_state, MODELS_OUT / 'classifier_v2s.pth')
    save_artifact(MODELS_OUT / 'classifier_v2s.pth', 'classifier_v2s.pth (peer)')
    print(f'\nPeer V2-S best val accuracy: {peer_best_acc:.4f}  ({peer_best_acc*100:.1f}%)')

    # Restore global best refs to B0 (server sees B0 as primary).
    best_acc = saved_best_acc
    best_state = saved_best_state
    net.load_state_dict(best_state)
    ema.ema.load_state_dict(best_state)

    del peer_train_loader, peer_val_loader, peer_train_ds, peer_val_ds
    del peer_sampler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print('\nTRAIN_ENSEMBLE_PEER=False → skipping V2-S peer phase.')
    peer_net = None

print('\n✓ Cell завершён. Ready for ONNX export (next cell).')


## 7. Экспорт ONNX + INT8 квантизация + OOD-калибровка

Сервер крутит классификатор на CPU 2-core VPS. Экспортируем:
- **B0 FP32 / INT8** (default classifier; primary)
- **V2-S FP32 / INT8** (опциональный peer; weighted softmax-blend на инференсе)
- **classes.json** с метриками + ensemble metadata + suggested NAP floor

Здесь же делается OOD-калибровка: grid по `NAP_CLASSIFIER_MSP_FLOOR` против
positive (val plants) и negative (val not_a_plant) сетов; ищется floor
максимизирующий (TNR − FPR). Результат пишется в `classes.json["ood_calibration"]`,
чтобы можно было вручную перенести в `server/pipeline.py`.

Ожидаемые размеры:

| Файл | Размер | Latency (240×240, batch=1, 2-core CPU) |
|------|--------|----------------------------------------|
| `classifier.pth` (PyTorch eager FP32) | ~22 MB | ~80-110 ms |
| `classifier.onnx` (ONNX FP32 B0)      | ~5 MB  | ~50-80 ms |
| `classifier_int8.onnx` (INT8 B0)      | ~3 MB  | ~25-40 ms |
| `classifier_v2s.onnx` (FP32 peer)     | ~85 MB | ~150-200 ms |
| `classifier_v2s_int8.onnx` (INT8 peer)| ~22 MB | ~80-120 ms |

Если оперативной памяти на T4 не хватает на peer — выключи `TRAIN_ENSEMBLE_PEER`
в начале cell 12.


In [ ]:
import copy, math
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_FP32 = MODELS_OUT / 'classifier.onnx'
ONNX_INT8 = MODELS_OUT / 'classifier_int8.onnx'
ONNX_PEER_FP32 = MODELS_OUT / 'classifier_v2s.onnx'
ONNX_PEER_INT8 = MODELS_OUT / 'classifier_v2s_int8.onnx'


def _export_to_onnx(model_to_export, dst: Path, img_size: int, label: str):
    export_model = copy.deepcopy(model_to_export).cpu().eval()
    dummy = torch.randn(1, 3, img_size, img_size, dtype=torch.float32)
    with torch.inference_mode():
        # opset 18 supports the latest GroupNorm / LayerNorm fast
        # paths and matches onnxruntime ≥ 1.17. Same numerics as 17
        # on EfficientNet (no GN), but future-proof for V2-S Fused.
        torch.onnx.export(
            export_model, dummy, str(dst),
            input_names=['input'], output_names=['logits'],
            dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
            opset_version=18, do_constant_folding=True,
            dynamo=False,
        )
    del export_model, dummy
    gc.collect()
    print(f'ONNX FP32 ({label}) → {dst}  ({dst.stat().st_size/1e6:.1f} MB)')

    m = onnx.load(str(dst))
    onnx.checker.check_model(m)
    try:
        from onnxsim import simplify
        m_simp, ok = simplify(m)
        if ok:
            onnx.save(m_simp, str(dst))
            print(f'  + onnxsim ({label}): simplified OK')
        else:
            print(f'  · onnxsim ({label}): ok=False, kept original')
    except Exception as e:
        print(f'  · onnxsim ({label}): skipped ({e})')
    del m
    gc.collect()


def _quantize_int8(src: Path, dst: Path, label: str):
    dst.unlink(missing_ok=True)
    quantize_dynamic(
        model_input=str(src),
        model_output=str(dst),
        weight_type=QuantType.QInt8,
        op_types_to_quantize=['MatMul'],
        per_channel=True,
    )
    print(f'ONNX INT8 ({label}) → {dst}  ({dst.stat().st_size/1e6:.1f} MB)')


_export_to_onnx(ema.ema, ONNX_FP32, IMG_SIZE, 'B0')
_quantize_int8(ONNX_FP32, ONNX_INT8, 'B0')
save_artifact(ONNX_FP32, 'classifier.onnx FP32')
save_artifact(ONNX_INT8, 'classifier_int8.onnx')

if peer_net is not None and peer_best_state is not None:
    _peer_export = build_v2s_net(num_classes)
    _peer_export.load_state_dict(peer_best_state)
    _peer_export.eval()
    _export_to_onnx(_peer_export, ONNX_PEER_FP32, PEER_IMG_SIZE, 'V2-S peer')
    _quantize_int8(ONNX_PEER_FP32, ONNX_PEER_INT8, 'V2-S peer')
    save_artifact(ONNX_PEER_FP32, 'classifier_v2s.onnx FP32')
    save_artifact(ONNX_PEER_INT8, 'classifier_v2s_int8.onnx')
    del _peer_export
    gc.collect()


def onnx_eval(path: Path, loader_):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    correct = total_n = 0
    for x, y in loader_:
        x_np = x.detach().cpu().numpy().astype(np.float32, copy=False)
        try:
            logits = sess.run(None, {inp: x_np})[0]
        except Exception as e:
            print(f'  ! {path.name}: inference error {e}')
            return float('nan')
        pred = np.argmax(logits, axis=1)
        correct += int((pred == y.detach().cpu().numpy()).sum())
        total_n += len(y)
    del sess
    gc.collect()
    return correct / total_n if total_n else float('nan')


print('\n── Sanity-check: PyTorch vs ONNX FP32 vs ONNX INT8 (B0) ──')
acc_fp32 = onnx_eval(ONNX_FP32, val_loader)
acc_int8 = onnx_eval(ONNX_INT8, val_loader)
print(f'\nTorch  EMA+TTA val acc : {b0_best_acc:.4f}')
print(f'ONNX FP32 (no TTA)     : {acc_fp32:.4f}')
print(f'ONNX INT8 (no TTA)     : {acc_int8:.4f}')

drift_fp32 = abs(acc_fp32 - b0_best_acc)
drift_int8 = abs(acc_int8 - acc_fp32)
if drift_fp32 > 0.04:
    print(f'  ⚠ FP32 drift {drift_fp32:.3f} > 0.04 — TTA contribution is high but suspicious')
if drift_int8 > 0.015:
    print(f'  ⚠ INT8 drift {drift_int8:.3f} > 0.015 — quantization degradation')
else:
    print(f'  ✓ INT8 drift {drift_int8:.3f} ≤ 0.015 — production-ready')

# v5.0: ensemble eval (B0 + V2-S softmax average) on val.
acc_ensemble = float('nan')
if peer_net is not None and ONNX_PEER_INT8.exists():
    print('\n── Ensemble (B0 INT8 + V2-S INT8 weighted softmax) ──')
    sess_b0 = ort.InferenceSession(str(ONNX_INT8), providers=['CPUExecutionProvider'])
    sess_pe = ort.InferenceSession(str(ONNX_PEER_INT8), providers=['CPUExecutionProvider'])
    inp_b0 = sess_b0.get_inputs()[0].name
    inp_pe = sess_pe.get_inputs()[0].name
    # Build a peer-resolution val loader once for the peer ONNX path.
    peer_eval_ds = datasets.ImageFolder(str(CLS_DIR / 'val'), transform=make_val_tf(PEER_IMG_SIZE))
    peer_eval_loader = DataLoader(
        peer_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=False,
    )
    correct = total_n = 0
    weight = PEER_WEIGHT_DEFAULT
    for (x_b0, y_b0), (x_pe, y_pe) in zip(val_loader, peer_eval_loader):
        # Both loaders walk val_ds in alphabetical order so y_b0 == y_pe.
        b0_logits = sess_b0.run(None, {inp_b0: x_b0.numpy().astype(np.float32, copy=False)})[0]
        pe_logits = sess_pe.run(None, {inp_pe: x_pe.numpy().astype(np.float32, copy=False)})[0]
        b0_logits = b0_logits - b0_logits.max(axis=1, keepdims=True)
        pe_logits = pe_logits - pe_logits.max(axis=1, keepdims=True)
        b0_p = np.exp(b0_logits); b0_p = b0_p / b0_p.sum(axis=1, keepdims=True)
        pe_p = np.exp(pe_logits); pe_p = pe_p / pe_p.sum(axis=1, keepdims=True)
        blend = (1.0 - weight) * b0_p + weight * pe_p
        pred = np.argmax(blend, axis=1)
        correct += int((pred == y_b0.numpy()).sum())
        total_n += int(y_b0.numel())
    acc_ensemble = correct / total_n if total_n else float('nan')
    print(f'Ensemble INT8 val acc  : {acc_ensemble:.4f}  (peer_weight={weight})')
    del sess_b0, sess_pe, peer_eval_loader, peer_eval_ds
    gc.collect()


# ── v5.0 OOD calibration sweep ──
# Goal: tune the two NAP gate floors used by server/pipeline.py
#   NAP_DETECTOR_CONF_FLOOR  (default 0.40)
#   NAP_CLASSIFIER_MSP_FLOOR (default 0.45)
# Strategy: build a synthetic NAP eval set from the train val/not_a_plant
# folder + held-back COCO frames, plus the *positive* set (all val
# samples whose ground-truth is NOT not_a_plant). Sweep the floors
# and report the (FPR on plants → NAP) and (TNR on NAP → NAP) trade-off.
print('\n── OOD calibration sweep ──')

ood_neg_dir = CLS_DIR / 'val' / 'not_a_plant'
ood_pos_paths = [
    Path(p) for (p, y) in val_ds.samples
    if class_names[y] != 'not_a_plant'
]
ood_neg_paths = list(ood_neg_dir.iterdir()) if ood_neg_dir.exists() else []
print(f'  positives (val, plants): {len(ood_pos_paths)}    '
      f'negatives (val, not_a_plant): {len(ood_neg_paths)}')

if ood_neg_paths and ood_pos_paths:
    sess_b0 = ort.InferenceSession(str(ONNX_INT8), providers=['CPUExecutionProvider'])
    inp_b0 = sess_b0.get_inputs()[0].name
    nap_idx_local = class_names.index('not_a_plant') if 'not_a_plant' in class_names else -1

    def _msp_for(path: Path) -> float:
        try:
            with Image.open(path) as im:
                im = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                arr = np.asarray(im, dtype=np.float32) / 255.0
                arr = (arr - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
                arr = arr.transpose(2, 0, 1)[None].astype(np.float32, copy=False)
            logits = sess_b0.run(None, {inp_b0: arr})[0][0]
            logits = logits - logits.max()
            p = np.exp(logits); p = p / p.sum()
            return float(p.max()), int(p.argmax())
        except Exception:
            return 0.0, -1

    pos_msps = [_msp_for(p) for p in ood_pos_paths]
    neg_msps = [_msp_for(p) for p in ood_neg_paths]
    del sess_b0
    gc.collect()

    grid_msp = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    print('  MSP floor  |  TNR on NAP (higher=better)  |  FPR on plants (lower=better)')
    rows: list[tuple[float, float, float]] = []
    for floor in grid_msp:
        # We approximate "OOD-gated to NAP" as: max-softmax-prob below
        # floor AND argmax != not_a_plant. (Detector floor needs raw
        # YOLO confidences; we leave it at the existing default of 0.40
        # since NAP is dominated by the classifier signal here.)
        tnr_nap = sum(
            1 for (msp, arg) in neg_msps
            if msp < floor and arg != nap_idx_local
        ) / max(len(neg_msps), 1)
        fpr_plants = sum(
            1 for (msp, arg) in pos_msps
            if msp < floor and arg != nap_idx_local
        ) / max(len(pos_msps), 1)
        rows.append((floor, tnr_nap, fpr_plants))
        print(f'    {floor:.2f}    |          {tnr_nap*100:5.1f}%           |        {fpr_plants*100:5.1f}%')
    # Pick the floor that maximises (TNR - FPR).
    best_row = max(rows, key=lambda r: r[1] - r[2])
    suggested_msp_floor = best_row[0]
    print(f'  ✓ Suggested NAP_CLASSIFIER_MSP_FLOOR = {suggested_msp_floor:.2f}  '
          f'(TNR={best_row[1]*100:.1f}%, FPR={best_row[2]*100:.1f}%)')
else:
    suggested_msp_floor = 0.45
    print('  · Skipped (need both positive and negative val samples)')

# ── Final classes.json with all v5.0 metadata ──
classes_meta = {
    '_comment': 'PlantScope v5.0 — calibrated houseplant classifier (EfficientNet-B0 @ 240, optional V2-S @ 300 ensemble).',
    'model_version': '5.0.0',
    'architecture': {
        'detector': 'YOLOv8n (single-class leaf)',
        'classifier': 'EfficientNet-B0 (torchvision, IMAGENET1K_V1)',
        'ensemble_peer': 'EfficientNetV2-S (torchvision, IMAGENET1K_V1)' if peer_net is not None else None,
    },
    'num_classes': num_classes,
    'class_names': class_names,
    'detector_classes': ['leaf'],
    'val_accuracy_torch_tta': round(b0_best_acc, 4),
    'val_accuracy_onnx_fp32': round(acc_fp32, 4) if not math.isnan(acc_fp32) else None,
    'val_accuracy_onnx_int8': round(acc_int8, 4) if not math.isnan(acc_int8) else None,
    'val_accuracy_ensemble_int8': (
        round(acc_ensemble, 4) if not math.isnan(acc_ensemble) else None
    ),
    'val_accuracy_peer_torch_tta': (
        round(peer_best_acc, 4) if peer_best_acc is not None else None
    ),
    'temperature': round(best_T, 4),
    'training': {
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'cutmix_prob': CUTMIX_PROB,
        'mixup_prob': MIXUP_PROB,
        'mixup_alpha': MIXUP_ALPHA,
        'label_smoothing': LABEL_SMOOTHING,
        'loss': (
            f'FocalCE(gamma={FOCAL_GAMMA}, label_smoothing={LABEL_SMOOTHING})'
            if USE_FOCAL_LOSS else
            f'CrossEntropy(label_smoothing={LABEL_SMOOTHING})'
        ),
        'sampler': 'WeightedRandomSampler (single balancing source)',
        'tta': '5-crop+hflip (final pass), hflip (val loop)',
        'ema_decay': EMA_DECAY,
        'grad_clip': GRAD_CLIP,
        'pv_cap_strategy': 'late_blight 800, bacterial/mosaic 700, other PV sources 500 each',
        'healthy_strategy': 'multi-crop top-3 NMS bboxes per houseplant photo',
        'pests_handling': 'NOT classes — surfaced as possible_pest_causes hints in UI',
        'augmentation': 'TrivialAugment + WarmLight + JPEG + RandomErasing + MotionBlur + RandomShadow',
        'calibration': 'temperature scaling on val NLL (extended grid 0.3-5.0)',
        'quantization': 'dynamic INT8 on MatMul only (Conv stays FP32)',
        'split': 'stratified per-class 15% val (min 8 samples), group-aware (iNat obs / multi-crop photo)',
        'early_stopping': f'patience {EARLY_STOP_PATIENCE} on Phase B',
        'sha256_dedup': True,
        'exif_strip': True,
        'deterministic': DETERMINISTIC,
    },
    'ensemble': {
        'peer_arch': 'EfficientNetV2-S' if peer_net is not None else None,
        'peer_img_size': PEER_IMG_SIZE,
        'peer_weight': PEER_WEIGHT_DEFAULT,
        '_note': 'Loaded by server/classifier.py if classifier_v2s*.onnx is present.',
    },
    'ood_calibration': {
        'suggested_classifier_msp_floor': float(suggested_msp_floor),
        '_note': 'Tune server/pipeline.py NAP_CLASSIFIER_MSP_FLOOR using this value.',
    },
}
(MODELS_OUT / 'classes.json').write_text(
    json.dumps(classes_meta, ensure_ascii=False, indent=2), encoding='utf-8')
save_artifact(MODELS_OUT / 'classes.json', 'classes.json')
print('\nclasses.json →', MODELS_OUT / 'classes.json')


## 8. End-to-end demo

YOLOv8 → bbox листа → классификатор (через ONNX INT8) → подпись.

Реализует ту же логику, что `server/pipeline.py`:
- size filter на bbox
- mean voting по нескольким кропам
- temperature scaling
- healthy fallback при низкой уверенности на болезни
- (v5.0) **5-crop + h-flip TTA** на финальной классификации
- (v5.0) **ensemble blend** B0 + V2-S softmax если peer-checkpoint доступен


In [ ]:
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

det = YOLO(str(MODELS_OUT / 'detector.pt'))
sess = ort.InferenceSession(
    str(MODELS_OUT / 'classifier_int8.onnx'),
    providers=['CPUExecutionProvider'],
)
inp_name = sess.get_inputs()[0].name
peer_sess = None
if (MODELS_OUT / 'classifier_v2s_int8.onnx').exists():
    peer_sess = ort.InferenceSession(
        str(MODELS_OUT / 'classifier_v2s_int8.onnx'),
        providers=['CPUExecutionProvider'],
    )
    peer_inp = peer_sess.get_inputs()[0].name

norm_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
norm_std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

TEMP = json.loads((MODELS_OUT / 'temperature.json').read_text(encoding='utf-8'))['temperature']

def _norm(arr):
    arr = arr.astype(np.float32) / 255.0
    return ((arr - norm_mean) / norm_std).transpose(2, 0, 1)[None]


def _five_crop(arr, size):
    H, W = arr.shape[:2]
    if H < size or W < size:
        return [arr]
    cy, cx = H // 2, W // 2
    crops = [
        arr[:size, :size],
        arr[:size, W - size:],
        arr[H - size:, :size],
        arr[H - size:, W - size:],
        arr[cy - size // 2:cy + size // 2, cx - size // 2:cx + size // 2],
    ]
    return crops


def _classify_tta(crop_pil):
    arr_full = np.array(crop_pil.convert('RGB'))
    H, W = arr_full.shape[:2]
    side = min(H, W)
    if side > IMG_SIZE * 1.5:
        # 5-crop+hflip when the bbox is comfortably bigger than IMG_SIZE.
        five = _five_crop(arr_full, IMG_SIZE)
        crops = []
        for c in five:
            c = np.array(Image.fromarray(c).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
            crops.append(c)
            crops.append(c[:, ::-1, :])  # h-flip
        x = np.concatenate([_norm(c) for c in crops], axis=0).astype(np.float32, copy=False)
    else:
        # Fallback: just centre + h-flip.
        a = np.array(crop_pil.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
        x = np.concatenate([_norm(a), _norm(a[:, ::-1, :])], axis=0).astype(np.float32, copy=False)
    logits = sess.run(None, {inp_name: x})[0]
    logits = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(logits / max(TEMP, 1e-3))
    p = p / p.sum(axis=1, keepdims=True)
    p = p.mean(axis=0)
    if peer_sess is not None:
        a_peer = np.array(crop_pil.convert('RGB').resize((PEER_IMG_SIZE, PEER_IMG_SIZE), Image.BILINEAR))
        x_peer = _norm(a_peer).astype(np.float32, copy=False)
        peer_logits = peer_sess.run(None, {peer_inp: x_peer})[0][0]
        peer_logits = peer_logits - peer_logits.max()
        pp = np.exp(peer_logits); pp = pp / pp.sum()
        p = (1.0 - PEER_WEIGHT_DEFAULT) * p + PEER_WEIGHT_DEFAULT * pp
    return p


# Pick a few demo images from val/.
demo = []
for cls in class_names[:6]:
    d = CLS_DIR / 'val' / cls
    if d.exists():
        for f in list(d.iterdir())[:1]:
            demo.append((cls, f))

cols = min(len(demo), 4)
rows = (len(demo) + cols - 1) // max(cols, 1)
plt.figure(figsize=(4 * cols, 4 * rows))
for i, (true_cls, path) in enumerate(demo, 1):
    img = Image.open(path).convert('RGB')
    arr = np.array(img)
    res = det.predict(arr, imgsz=640, conf=0.4, verbose=False, device=device)
    boxes = res[0].boxes.xyxy.cpu().numpy().tolist() if res and res[0].boxes is not None else []
    if boxes:
        x1, y1, x2, y2 = (int(round(v)) for v in boxes[0])
        crop = img.crop((x1, y1, x2, y2))
    else:
        crop = img
    p = _classify_tta(crop)
    pred = class_names[int(np.argmax(p))]
    conf = float(p.max())
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(f'true={true_cls}\npred={pred} ({conf:.2f})')
    plt.axis('off')
plt.tight_layout()
plt.show()


## 9. Скачать веса

Положите в `server/models/`:
- `detector.pt`               — YOLOv8n PyTorch
- `detector.onnx`             — YOLOv8n ONNX (production)
- `classifier.pth`            — EfficientNet-B0 PyTorch (fallback)
- `classifier.onnx`           — EfficientNet-B0 ONNX FP32
- `classifier_int8.onnx`      — EfficientNet-B0 INT8 ⭐ default for server
- `classifier_v2s.pth`        — EfficientNetV2-S PyTorch (peer; опционально)
- `classifier_v2s.onnx`       — EfficientNetV2-S ONNX FP32 (peer)
- `classifier_v2s_int8.onnx`  — EfficientNetV2-S INT8 (peer; auto-загружается сервером)
- `classes.json`              — таксономия + метрики + ensemble metadata + OOD calibration
- `temperature.json`          — параметр калибровки
- `MODEL_CARD.md`             — генерируется автоматически в последней ячейке

Артефакты также автоматически сохранены в `/content/drive/MyDrive/PlantScope_v50/`.


In [ ]:
artifacts = [
    MODELS_OUT / 'detector.pt',
    MODELS_OUT / 'detector.onnx',
    MODELS_OUT / 'classifier.pth',
    MODELS_OUT / 'classifier.onnx',
    MODELS_OUT / 'classifier_int8.onnx',
    MODELS_OUT / 'classifier_v2s.pth',
    MODELS_OUT / 'classifier_v2s.onnx',
    MODELS_OUT / 'classifier_v2s_int8.onnx',
    MODELS_OUT / 'classes.json',
    MODELS_OUT / 'temperature.json',
    MODELS_OUT / 'MODEL_CARD.md',
]
print('Артефакты в', MODELS_OUT, ':')
for p in artifacts:
    if p.exists():
        print(f'  {p.name:<28} {p.stat().st_size / 1e6:>6.2f} MB')
    else:
        print(f'  {p.name:<28} (missing — пропускаем)')

if BACKUP_DIR is not None:
    print(f'\nBackup-копии:  {BACKUP_DIR}')

print(
    '\nДальше: скопируй *.pt / *.onnx / classes.json / temperature.json / '
    'MODEL_CARD.md из MODELS_OUT в server/models/ — сервер сам подхватит.'
)


## 10. MODEL_CARD.md (auto-generated)

Финальная ячейка собирает MODEL_CARD из реальных метрик прогона
(val accuracy B0 / V2-S / ensemble, temperature, suggested OOD floor,
размеры артефактов, hyperparameters) и сохраняет рядом с весами.
Используется в `server/models/MODEL_CARD.md` после переноса.


In [ ]:
from datetime import datetime, timezone

# Derive a few sizes for the report (None if file missing).
def _size_mb(p: Path) -> str:
    if p.exists():
        return f'{p.stat().st_size/1e6:.2f} MB'
    return '—'

card = f'''# PlantScope v5.0 — Auto-generated MODEL_CARD

Generated: {datetime.now(timezone.utc).isoformat(timespec="seconds")}

## Summary

Two-stage houseplant pipeline. Stage 1 = YOLOv8n single-class leaf
detector. Stage 2 = EfficientNet-B0 (primary) + optional V2-S peer in
weighted softmax ensemble.

## Classes (sorted alphabetical, matches `ImageFolder.classes`)

{chr(10).join(f"- {c}" for c in class_names)}

## Held-out val metrics

| Path | top-1 |
|------|-------|
| Torch B0 (EMA + h-flip TTA) | {b0_best_acc:.4f} |
| ONNX B0 FP32 (no TTA) | {acc_fp32:.4f} |
| ONNX B0 INT8 (no TTA) | {acc_int8:.4f} |
| ONNX V2-S Peer (Torch, EMA) | {peer_best_acc if peer_best_acc is not None else "—"} |
| Ensemble B0+V2-S INT8 (peer_weight={PEER_WEIGHT_DEFAULT}) | {acc_ensemble if not math.isnan(acc_ensemble) else "—"} |

Temperature: {best_T:.3f}  (val NLL {raw_nll:.4f} → {best_nll:.4f})

## Artefact sizes

| File | Size |
|------|------|
| `detector.pt` | {_size_mb(MODELS_OUT / "detector.pt")} |
| `detector.onnx` | {_size_mb(MODELS_OUT / "detector.onnx")} |
| `classifier.pth` | {_size_mb(MODELS_OUT / "classifier.pth")} |
| `classifier.onnx` | {_size_mb(MODELS_OUT / "classifier.onnx")} |
| `classifier_int8.onnx` | {_size_mb(MODELS_OUT / "classifier_int8.onnx")} |
| `classifier_v2s.pth` | {_size_mb(MODELS_OUT / "classifier_v2s.pth")} |
| `classifier_v2s.onnx` | {_size_mb(MODELS_OUT / "classifier_v2s.onnx")} |
| `classifier_v2s_int8.onnx` | {_size_mb(MODELS_OUT / "classifier_v2s_int8.onnx")} |

## Training config (this run)

- IMG_SIZE: {IMG_SIZE} (peer {PEER_IMG_SIZE})
- BATCH_SIZE: {BATCH_SIZE} (peer {PEER_BATCH_SIZE})
- LABEL_SMOOTHING: {LABEL_SMOOTHING}
- CUTMIX_PROB / MIXUP_PROB: {CUTMIX_PROB} / {MIXUP_PROB}
- EMA decay: {EMA_DECAY}
- Loss: {"FocalCE γ=" + str(FOCAL_GAMMA) if USE_FOCAL_LOSS else "CrossEntropy(label_smoothing)"}
- Train ensemble peer: {TRAIN_ENSEMBLE_PEER}
- Deterministic: {DETERMINISTIC}
- Resume from Drive: {RESUME_FROM_DRIVE}

## OOD calibration

- suggested NAP_CLASSIFIER_MSP_FLOOR: {suggested_msp_floor:.2f}
  (set in `server/pipeline.py`)

## Class-balance counts (train split)

| Class | Train count |
|-------|-------------|
{chr(10).join(f"| {c} | {n} |" for c, n in zip(class_names, counts))}

## Reproduction

1. Open notebook in Colab → T4 GPU runtime.
2. Upload `kaggle.json`.
3. Run All cells from top.
4. Copy `out/*` into `server/models/`.

## Risks / known limitations

- Disease classes overlap visually on small datasets — confidence
  gates in `server/pipeline.py` downgrade weak verdicts to healthy or
  not_a_plant rather than asserting a disease.
- `mosaic_virus` and `iron_chlorosis` overlap on yellowing leaves;
  expect occasional cross-confusion in the per-class F1.
- `late_blight` proxy data is field-crop (PV tomato/potato) — visual
  transfer to indoor ornamentals is reasonable but imperfect.
'''

(MODELS_OUT / 'MODEL_CARD.md').write_text(card, encoding='utf-8')
save_artifact(MODELS_OUT / 'MODEL_CARD.md', 'MODEL_CARD.md')
print('MODEL_CARD.md →', MODELS_OUT / 'MODEL_CARD.md')
print('--- card preview ---')
print(card[:1200])
